# Paper figures: one notebook, one cell per figure

Every figure of the paper *Calling a Draw* is drawn by one code cell below. To change a text, a colour or a size, edit the
cell and run it again; the figure is written as PDF and PNG into `OUT` (default `VLM/paper_figures_nb/figs_v3`).

**How to run.** On bjserver, with the kernel of `/media/shawn/C/papers/VLM/.fignb` (a virtual environment that inherits
the packages of `lis-omni` and adds Jupyter; `lis-omni` itself is not modified):
`/media/shawn/C/papers/VLM/.fignb/bin/jupyter lab` or, without a browser,
`.fignb/bin/jupyter nbconvert --to notebook --execute --output /tmp/run.ipynb paper_figures.ipynb`.
Run the **Setup** cell first. Each figure cell can then be run on its own, in any order; Figures 1 and 2 also need the
cell *Data for Figures 1 and 2*. The cells marked *light* (architecture, cross-model, window) also run on the minipc in
`VLM/.figenv`; the others need the judge outputs and images that exist only on bjserver.

**Keeping the history.** This notebook lives in the code repository (`EAGLE-VLM/paper_figures/paper_figures.ipynb`).
Clear the outputs before a commit so that the diff shows the code only:
`.fignb/bin/jupyter nbconvert --clear-output --inplace paper_figures.ipynb`.

| Cell | Paper figure | File written |
|---|---|---|
| Figure 1 | teaser: four situations | `F1_teaser` |
| Figure 2 | motivating experiment | `F2_motivation` |
| Figure 3 | mechanism: the judge's output space | `F3_phase_imagenhub_edit` |
| Cross-model | Sec. 4.1, judge by judge | `F7_cross_model` + table rows |
| Failure kinds (small / full) | Sec. 4.3 and supplement | `F5s_failure_kinds_small`, `F5_failure_kinds` |
| Architecture | method figure | `A1_architecture` |
| Window | Sec. 6, measured and predicted window | `T1s_window_small` |
| Attention (registers / sinks / first probe) | supplement | `F6_attention_registers`, `F6_attention_sinks`, `F4_attention` |
| Tie and similarity | supplement | `F5_tie_is_similarity`, `tie_similarity.json` |

The older supplement figures (`Fig2`--`Fig5`, `D*`, `T1_interval_theory`) are still produced by the notebooks
`01`--`08` in `VLM/paper_figures_nb` (generated by `build_notebooks.py`).

## Setup (run first)

In [ ]:
import os, sys
import matplotlib, matplotlib.pyplot as plt
VLM = next(p for p in ("/media/shawn/C/papers/VLM", "/home/shawn/bjserver-papers/VLM") if os.path.isdir(p))   # bjserver, else minipc
REPO = "/media/shawn/C/papers/EAGLE-VLM"                    # code repository (bjserver only)
NB = os.path.join(VLM, "paper_figures_nb")                  # figstyle.py, data/ (small tables), figure folders
OUT = os.environ.get("FIG_OUT", os.path.join(NB, "figs_v3"))   # where the figures are written
TABLE_ROWS = os.environ.get("FIG_TABLE", os.path.join(VLM, "paper_vA_standard/tables/tab_cross_model.tex"))   # rows of the cross-model table
ATTN_DIR = os.path.join(VLM, "agent_reward_bench/attn_20261114"); ATTN2_DIR = os.path.join(VLM, "agent_reward_bench/attn2_20261114")   # attention probes
os.makedirs(OUT, exist_ok=True); sys.path.insert(0, NB)
if os.path.isfile(os.path.join(REPO, 'paper_figures/figstyle.py')): sys.path.insert(0, os.path.join(REPO, 'paper_figures'))   # the tracked copy of the shared style
if os.path.isdir(REPO): os.chdir(REPO)                      # the repository code reads configs/ relative to its root
import figstyle; figstyle.FIGS = OUT; figstyle.DATA = os.path.join(NB, 'data')   # small data tables stay outside the repository
def fresh(rc=None):
    """reset matplotlib's settings so that one figure cell cannot change the look of another"""
    matplotlib.rcdefaults()
    if rc: plt.rcParams.update(rc)
print("figures ->", OUT, "| repository present:", os.path.isdir(REPO))

## Data for Figures 1 and 2
One judge (Qwen3-VL-8B) on ImagenHub edit: its answers under the three questions and the human labels. Colours (`C`) and fonts (`RC_V3`) of Figures 1 and 2 are set here.

In [ ]:
"""Prototype figures for the CV-style redesign (2026-11-14).  Run on bjserver (images are there).
F1  teaser: real ImagenHub edit pairs that people rated as ties (both edits failed), with what one judge says about them.
    Selection rule (no hand-picking): human tie with both ratings 0; the forced choice names different candidates in the two
    orders, each with probability >= .8; sorted by crc32 of the pair id; the first pairs with distinct source images.
F2  motivation experiment, one judge (Qwen3-VL-8B), all ImagenHub edit pairs: (a) what the judge answers on human-tied pairs
    under three questions; (b) the probability it gives the tie answer, tied vs decided pairs; (c) the same after one
    weight per answer (weights from the label shares of a 25% fit split of the sources; shown on the other sources).
usage (repo root): python fig_v3.py <out_dir>"""
import sys, os, zlib, json
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
sys.path.insert(0, "/media/shawn/C/papers/EAGLE-VLM"); sys.path.insert(0, "/media/shawn/C/papers/EAGLE-VLM/scripts")
import absolute_quad as AQ
from eaglevlm.pipeline.certify import source_key
J = "qwen3vl8b"; SPEC = "imagenhub/edit"
C = dict(fail="#d9541e", ok="#1a9f6e", ours="#1f6fd0", grey="#8a8882", light="#d9d7cf", ink="#111111", ink2="#4a4946")
RC_V3 = dict({"font.family": "serif", "font.serif": ["STIXGeneral", "DejaVu Serif"], "mathtext.fontset": "stix", "font.size": 7.5, "axes.titlesize": 8,
                     "axes.labelsize": 7.5, "xtick.labelsize": 6.8, "ytick.labelsize": 6.8, "legend.fontsize": 6.8, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.linewidth": .6, "xtick.major.width": .6, "ytick.major.width": .6, "pdf.fonttype": 42, "savefig.dpi": 300, "legend.frameon": False})
ROOT = ["/media/shawn/C/papers/VLM/judgebench_local", "/media/shawn/C/papers/VLM/judgebench"]
def img(p, side=320):
    for r in ROOT:
        q = os.path.join(r, p)
        if os.path.exists(q):
            im = Image.open(q).convert("RGB"); w, h = im.size; s = min(w, h); im = im.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s)); return im.resize((side, side), Image.LANCZOS)
    raise FileNotFoundError(p)
lit = AQ.__dict__.get("lit") or (lambda v: __import__("ast").literal_eval(v) if isinstance(v, str) else v)

cfg = AQ.load_cfg(f"configs/exp/{AQ.SPECS[SPEC]}.yaml"); P_ = cfg["paths"]
st = AQ.Store(P_["items"], P_["outputs"], P_.get("items_override"), P_.get("snapshot")); ds, I = AQ.dataset_items(st, SPEC)
S = AQ.signals(st, ds, J, I); ids = sorted(S); PW = AQ.pointwise_feats(st, ds, J, ids)
f1 = np.array([S[i][0] for i in ids]); f2 = np.array([S[i][1] for i in ids]); Q = (np.array([S[i][2] for i in ids]) + np.array([S[i][3] for i in ids])) / 2; Q = Q / Q.sum(1, keepdims=True)
T1 = np.array([np.asarray(S[i][2]).argmax() for i in ids]); T2 = np.array([np.asarray(S[i][3]).argmax() for i in ids])
y = np.array([{"0": 0, "1": 1, "tie": 2}[str(I[i]["label"])] for i in ids]); R = np.array([AQ.ratings(SPEC, I[i]) for i in ids], float); tie = y == 2
src = np.array([source_key(I[i]) for i in ids])

EDIT = dict(C=C, RC_V3=RC_V3, ROOT=ROOT, img=img, lit=lit, st=st, ds=ds, I=I, S=S, ids=ids, PW=PW, f1=f1, f2=f2, Q=Q, T1=T1, T2=T2, y=y, R=R, tie=tie, src=src, J=J, SPEC=SPEC)   # snapshot: the figure cells restore these names
print(len(ids), "pairs,", int(tie.sum()), "human ties")

## Figure 1: teaser, four situations
The four pairs are chosen by the rules in the comment (first in a hashed order). Texts to edit: `HEAD` (panel titles), `NOTE` (the two italic lines under each panel).

In [ ]:
fresh(EDIT['RC_V3']); globals().update(EDIT)
# ---------------------------------------------------------------- F1 teaser: four situations, one pair each
# Rules (no hand-picking; the first pair in crc32 order of the pair id, distinct sources across panels):
#  (1) decided: one edit with the highest human rating, the other with the lowest; no condition on the judge.
#  (2) human tie, both ratings 0, and "tie" is the judge's answer to the tie question in both orders.
#  (3) human tie, both ratings 0, two confident (>= .8) and opposite forced choices, "tie" answered in neither order.
#  (4) human tie, both ratings at the top of the scale; no condition on the judge.
crc = np.argsort([zlib.crc32(i.encode()) for i in ids]); okpw = ~np.isnan(PW[:, :2]).any(1)
flip = (f1 > .5) != (f2 > .5); conf = np.minimum(np.maximum(f1, 1 - f1), np.maximum(f2, 1 - f2)) >= .8
TT = (T1 == 2) & (T2 == 2); anyT = (T1 == 2) | (T2 == 2); RMAX = np.nanmax(R); bf = tie & (R.max(1) == 0)
M = [(~tie) & (R.max(1) == RMAX) & (R.min(1) == 0) & okpw, bf & TT & okpw, bf & flip & conf & ~anyT & okpw, tie & (R.min(1) == RMAX) & okpw]
def px(p):
    for r in ROOT:
        q = os.path.join(r, p)
        if os.path.exists(q): return np.asarray(Image.open(q).convert("RGB").resize((128, 128), Image.BILINEAR), float) / 255
def change(k):          # mean absolute RGB difference of each edit from the source, and between the two edits (share of the pixel range)
    x = I[ids[k]]; s_ = px(lit(x["context"])[0]); a_ = px(lit(x["cand"])[0]); b_ = px(lit(x["cand"])[1]); return np.abs(a_ - s_).mean(), np.abs(b_ - s_).mean(), np.abs(a_ - b_).mean()
seen = set(); pick = []
for m in M:
    for k in crc:
        if m[k] and src[k] not in seen:
            pick.append(k); seen.add(src[k]); break
print("F1 candidates per panel", [int(m.sum()) for m in M], "| judge flips on panel-1 candidates %.3f | says tie in both orders: all pairs %.4f (n %d), of which human ties %.3f | in either order: tied %.4f decided %.4f | both-max ties: flip %.3f, tie answered %.3f"
      % (flip[M[0]].mean(), TT.mean(), TT.sum(), tie[TT].mean(), anyT[tie].mean(), anyT[~tie].mean(), flip[M[3]].mean(), anyT[M[3]].mean()))
HEAD = ["(1) A winner exists: the judge finds it", "(2) Both fail, unchanged: the judge says “tie”", "(3) Both fail: the judge still names a winner", "(4) Both succeed: either choice is right"]
fig = plt.figure(figsize=(6.875, 4.12)); gs = fig.add_gridspec(2, 2, left=.005, right=.995, top=.995, bottom=.012, wspace=.035, hspace=.075)
for n, k in enumerate(pick):
    x = I[ids[k]]; ctx = lit(x["context"]); cd = lit(x["cand"]); sub = gs[n // 2, n % 2].subgridspec(3, 3, height_ratios=[.21, 1, .60], wspace=.03, hspace=.02)
    dec = not tie[k]; win = int(R[k, 1] > R[k, 0]) if dec else -1; good = [R[k, c] == RMAX for c in (0, 1)]; col = C["ok"] if dec else C["fail"]
    ax = fig.add_subplot(sub[0, :]); ax.axis("off"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.text(0, .78, HEAD[n], fontsize=7.6, fontweight="bold", color=col, va="center")
    ax.text(1, .78, (f"people rate {R[k, 0]:g} and {R[k, 1]:g}: kept" if dec else f"people rate {R[k, 0]:g} and {R[k, 1]:g}: tie, removed"), fontsize=6.3, color=col, va="center", ha="right")
    ins = x["question"].strip(); ins = ins if len(ins) <= 60 else ins[:58] + "…"; ax.text(.004, .2, "instruction: “" + ins + "”", fontsize=6.9, style="italic", color=C["ink"], va="center")
    for c, (p, t) in enumerate(((ctx[0], "source"), (cd[0], "edit A"), (cd[1], "edit B"))):
        ax = fig.add_subplot(sub[1, c]); ax.imshow(img(p)); ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_visible(True); sp.set_linewidth(.5 if c == 0 else 1.3); sp.set_color(C["light"] if c == 0 else (C["ok"] if good[c - 1] else C["fail"]))
        ax.text(.03, .97, t if c == 0 else t + (": succeeded" if good[c - 1] else ": failed"), transform=ax.transAxes, va="top", ha="left", fontsize=6.2, color="white", bbox=dict(boxstyle="round,pad=.15", fc=(0, 0, 0, .55), ec="none"))
    ax = fig.add_subplot(sub[2, :]); ax.axis("off"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    a1 = "A" if f1[k] > .5 else "B"; a2 = "A" if f2[k] > .5 else "B"     # f2: probability that candidate A is better when it is shown second
    tp = "below 0.01" if Q[k, 2] < .005 else f"{Q[k, 2]:.2f}"; cA, cB, cAB = change(k)
    ax.text(0, .90, f"Judge, “Which is better?”   A shown first: {a1} ({max(f1[k], 1 - f1[k]):.2f})   B shown first: {a2} ({max(f2[k], 1 - f2[k]):.2f})   " + ("→ flips" if a1 != a2 else "→ same"), fontsize=6.6, color=C["ink"], va="center")
    ax.text(0, .71, f"Judge, “…or a tie?”   answers: {'ABT'[T1[k]]} and {'ABT'[T2[k]]}   probability of “tie”: {tp}", fontsize=6.6, color=C["ink"], va="center")
    ax.text(0, .52, f"Judge, score for each edit (1–9):   A {PW[k, 0]:.1f}   B {PW[k, 1]:.1f}", fontsize=6.6, color=C["ours"], va="center")
    fl = "names a different edit in each order" if a1 != a2 else "names the same edit in both orders"
    NOTE = [f"The judge names the edit people preferred, whichever edit comes first,\nand its two scores are far apart. Pairs like this one are what benchmarks keep.",
            f"Both edits return the source almost unchanged (they differ from it by {100 * cA:.0f}% and {100 * cB:.0f}%).\nForced to choose, the judge {fl}; offered a tie, it takes it.",
            f"There is no better edit. The judge {fl}, with confidence,\nand does not take the tie it is offered. Its scores agree with people: both failed.",
            f"People rate both edits a success: also a tie, also removed. The two edits differ by only {100 * cAB:.0f}%,\nyet “tie” gets no probability. The judge {fl}; either is right."][n]
    ax.text(0, .17, NOTE, fontsize=6.4, color=C["ink2"], va="center", style="italic", linespacing=1.25)
    print("pick", n + 1, ids[k], "R", R[k], "f1", round(float(f1[k]), 3), "f2", round(float(f2[k]), 3), "T", T1[k], T2[k], "Q", np.round(Q[k], 4), "PW", np.round(PW[k, :2], 2), "change A, B, A-vs-B", np.round([cA, cB, cAB], 3))
fig.savefig(f"{OUT}/F1_teaser.pdf"); fig.savefig(f"{OUT}/F1_teaser.png", dpi=220)
json.dump([ids[k] for k in pick], open(f"{OUT}/F1_pairs_four.json", "w"))

plt.show()

## Figure 2: motivating experiment
(a) three questions; (b) ties dropped; (c) ties kept, as given; (d) after one weight per answer. Panel titles are in `PAN`, the row texts of (a) in `rows`.

In [ ]:
fresh(EDIT['RC_V3']); globals().update(EDIT)
from matplotlib.patches import Rectangle
# ---------------------------------------------------------------- F2 motivation
from sklearn.metrics import roc_auc_score
ug, gi = np.unique(src, return_inverse=True); rng = np.random.default_rng(0); perm = rng.permutation(len(ug)); fit = np.isin(gi, perm[: len(ug) // 4]); rest = ~fit
pi = np.bincount(y[fit], minlength=3) / fit.sum(); w = np.ones(3)
for _ in range(300):
    P = Q * w; P /= P.sum(1, keepdims=True); w = w * pi / np.maximum(P.mean(0), 1e-9); w /= w.sum()
Pc = Q * w; Pc /= Pc.sum(1, keepdims=True); wr = w / w[:2].mean()
says = ((T1 == 2).astype(float) + (T2 == 2)) / 2; low = np.nanmax(PW[:, :2], 1) < 5
GT, GD = C["fail"], C["grey"]
fig = plt.figure(figsize=(6.875, 2.3)); g = fig.add_gridspec(1, 5, width_ratios=[.9, .16, 1, 1, 1], left=.095, right=.988, top=.80, bottom=.2, wspace=.09)
fig.legend(handles=[Rectangle((0, 0), 1, 1, color=GT, alpha=.85), Rectangle((0, 0), 1, 1, color=GD, alpha=.85)],
           labels=[f"pairs that people call a tie ({tie.sum():,})", f"pairs in which people prefer one edit ({(~tie).sum():,})"], loc="upper center", ncol=2, bbox_to_anchor=(.5, 1.01), handlelength=1.2, columnspacing=2.2, fontsize=7)
a = fig.add_subplot(g[0]); rows = [("“Which is\nbetter?”", "the answer flips with the order", flip), ("“…or a tie?”", "the judge answers “tie”", says), ("“How good\nis each?”", "both edits are scored below 5", low)]
for r, (nm, what, v) in enumerate(rows):
    yy = 2 - r; vt, vd = 100 * np.nanmean(v[tie]), 100 * np.nanmean(v[~tie])
    a.barh(yy + .0, vt, height=.26, color=GT); a.barh(yy - .29, vd, height=.26, color=GD); a.text(0, yy + .2, what, va="bottom", ha="left", fontsize=6.4, style="italic", color=C["ink"])
    a.text(vt + 2, yy + .0, f"{vt:.0f}%", va="center", fontsize=6.5, color=C["ink"]); a.text(vd + 2, yy - .29, f"{vd:.0f}%", va="center", fontsize=6.5, color=C["ink2"])
a.set_yticks([1.855, .855, -.145]); a.set_yticklabels([r[0] for r in rows], fontsize=6.6, linespacing=1.15, fontweight="bold"); a.set_xlim(0, 100); a.set_xticks([0, 50, 100]); a.set_ylim(-.5, 2.5); a.set_xlabel("% of the pairs in each group")
a.set_title("(a) Three questions", loc="left", x=-.42); a.spines["left"].set_visible(False); a.tick_params(axis="y", length=0)
lgt = lambda p: np.log(np.clip(p, 1e-10, 1) / np.clip(1 - p, 1e-10, 1)); bins = np.linspace(-21.5, 16.5, 55); ax0 = None
e_ = int(np.floor(np.log10(w[2] / w[0]))); yy = tie[rest]; tk = [np.log(1e-8), np.log(1e-4), 0, -np.log(1e-4)]
PAN = [(Q[rest, 2], False, "(b) Ties dropped"), (Q[rest, 2], True, "(c) Ties kept, as given"), (Pc[rest, 2], True, "(d) After one weight per answer")]
for k, (P, keep, t) in enumerate(PAN):
    a = fig.add_subplot(g[k + 2], sharey=ax0); ax0 = ax0 or a; ht = np.histogram(lgt(P[yy]), bins=bins)[0] / yy.sum(); hd = np.histogram(lgt(P[~yy]), bins=bins)[0] / (~yy).sum()
    a.stairs(100 * hd, bins, fill=True, color=GD, alpha=.75)
    if keep: a.stairs(100 * ht, bins, fill=True, color=GT, alpha=.65)
    a.axvline(0, color=C["ink"], lw=.8, ls=(0, (3, 2))); a.set_title(t, loc="left", fontsize=7.6, x=-.04); a.set_xlim(-21.5, 16.5); a.set_ylim(0, 21)
    a.set_xticks(tk); a.set_xticklabels(["$10^{-8}$", "$10^{-4}$", "½", "$1{-}10^{-4}$"], fontsize=6.2)
    a.text(.5, 20.7, "“tie” is the\nanswer →", fontsize=5.7, va="top", ha="left", color=C["ink"], linespacing=1.15)
    td, tt_ = 100 * np.mean(P[~yy] > .5), 100 * np.mean(P[yy] > .5)
    if not keep: txt = f"{100 * tie.mean():.0f}% of the pairs\nare removed.\n“tie” is the answer\non {td:.0f}% of the rest:\nnothing looks wrong"
    else:
        auc = roc_auc_score(yy, P); txt = (f"the groups separate:\nAUROC {auc:.2f}\n" if k == 1 else f"AUROC {auc:.2f}\n(unchanged)\n") + f"“tie” is the answer\non {tt_:.0f}% of tied,\n{td:.0f}% of decided"
        if k == 2: txt = "each probability ×\none constant, set on\n¼ of the sources;\n" + txt
        print("panel", "bcd"[k], "AUROC %.3f | overlap %.3f | median tie prob tied %.2e decided %.2e | above .5: tied %.3f decided %.3f" % (auc, np.minimum(ht, hd).sum(), np.median(P[yy]), np.median(P[~yy]), np.mean(P[yy] > .5), np.mean(P[~yy] > .5)))
    a.text(.02, .97, txt, transform=a.transAxes, ha="left", va="top", fontsize=5.9, color=C["ink"], linespacing=1.22)
    if k == 0: a.set_ylabel("% of the pairs in each group", labelpad=2)
    else: plt.setp(a.get_yticklabels(), visible=False)
    if k == 1:
        m0 = lgt(np.median(P[yy])); a.annotate("", xy=(m0 + np.log(w[2] / w[0]), 7.0), xytext=(m0, 7.0), arrowprops=dict(arrowstyle="->", color=C["ours"], lw=.9))
        a.text(m0 + np.log(w[2] / w[0]) / 2 - 1, 7.3, "the weights shift\nevery pair →", ha="center", va="bottom", fontsize=5.7, color=C["ours"], linespacing=1.1)
    if k == 1: a.set_xlabel("probability the judge gives to the answer “tie” (logit scale); in (d), after the weights", fontsize=7)
fig.savefig(f"{OUT}/F2_motivation.pdf"); fig.savefig(f"{OUT}/F2_motivation.png", dpi=220)
print("weights relative to A:", np.round(w / w[0], 2), "label shares:", np.round(pi, 3), "mean raw probabilities (A, B, T): fit split", np.round(Q[fit].mean(0), 4), "tied pairs", np.round(Q[tie].mean(0), 3), "decided pairs", np.round(Q[~tie].mean(0), 3))
print("(a): flips tied %.3f decided %.3f | says tie (share of the two orders) tied %.4f decided %.4f | both scores < 5 tied %.3f decided %.3f" % (flip[tie].mean(), flip[~tie].mean(), says[tie].mean(), says[~tie].mean(), np.nanmean(low[tie]), np.nanmean(low[~tie])))
plt.show()

## Figure 3: the judge's output space
Panel titles are in the `set_title` / `phase(...)` calls; colours in `COL`, legend names in `NAME`.

In [ ]:
fresh()
"""Prototype (2026-11-14, the author's idea): one judge's pairs as points in the space of its own outputs, seen from several sides.
Axes: d = order-free preference for candidate A (mean logit of the forced choice over the two orders); t = log-odds of the
tie answer against the stronger of A and B (A/B/tie prompt, order-averaged); s = mean pointwise score of the two candidates.
Colour = what people said: A preferred, B preferred, tie with both rated low, tie with both rated high, other tie.
Views: (a) the A-versus-B axis alone with the tie probability on a linear scale (what a forced-choice read-out shows);
(b) the 3D cloud; (c) the d-t plane as a phase map (majority human label per cell); (d) the t-s plane.
usage (repo root): python fig_phase.py <out_dir> [spec] [judge]"""
import sys, os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
sys.path.insert(0, "/media/shawn/C/papers/EAGLE-VLM"); sys.path.insert(0, "/media/shawn/C/papers/EAGLE-VLM/scripts")
import absolute_quad as AQ, branch_probes as bp
SPEC = "imagenhub/edit"; J = "qwen3vl8b"      # benchmark and judge shown
plt.rcParams.update({"font.family": "serif", "font.serif": ["STIXGeneral", "DejaVu Serif"], "mathtext.fontset": "stix", "font.size": 7.5, "axes.titlesize": 7.6, "axes.labelsize": 7.5,
                     "xtick.labelsize": 6.8, "ytick.labelsize": 6.8, "legend.fontsize": 6.8, "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": .6, "pdf.fonttype": 42, "legend.frameon": False})
COL = {"A": "#1f6fd0", "B": "#7a52c7", "fail": "#d9541e", "good": "#1a9f6e", "other": "#c9a227"}
NAME = {"A": "people prefer A", "B": "people prefer B", "fail": "tie: both rated low", "good": "tie: both rated high", "other": "tie: other"}
cfg = AQ.load_cfg(f"configs/exp/{AQ.SPECS[SPEC]}.yaml"); P_ = cfg["paths"]
st = AQ.Store(P_["items"], P_["outputs"], P_.get("items_override"), P_.get("snapshot")); ds, I = AQ.dataset_items(st, SPEC)
S = AQ.signals(st, ds, J, I); ids = sorted(S); PW = AQ.pointwise_feats(st, ds, J, ids); ok = ~np.isnan(PW[:, :2]).any(1); ids = [i for i, o in zip(ids, ok) if o]; PW = PW[ok]
lg = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / np.clip(1 - p, 1e-6, 1))
f1 = np.array([S[i][0] for i in ids]); f2 = np.array([S[i][1] for i in ids]); Q = (np.array([S[i][2] for i in ids]) + np.array([S[i][3] for i in ids])) / 2; Q = Q / Q.sum(1, keepdims=True)
d = (lg(f1) + lg(f2)) / 2; t = np.log(np.maximum(Q[:, 2], 1e-12) / np.maximum(Q[:, 0], Q[:, 1])); s = PW[:, :2].mean(1)
tt = [bp.tie_type(SPEC, I[i]) for i in ids]; lab = np.array([{"0": "A", "1": "B"}.get(str(I[i]["label"])) or ("fail" if ty == "both bad" else "good" if ty == "both good" else "other") for i, ty in zip(ids, tt)])
rng = np.random.default_rng(0); sub = rng.choice(len(ids), min(len(ids), 3500), replace=False)
print(SPEC, J, len(ids), {k: float(np.mean(lab == k)) for k in COL}, "t range", np.percentile(t, [1, 50, 99]).round(1), "d range", np.percentile(d, [1, 50, 99]).round(1))
NAME = {"A": "people prefer A", "B": "people prefer B", "fail": "tie, both rated low", "good": "tie, both rated high", "other": "tie, other ratings"}
istie = np.isin(lab, ["fail", "good", "other"]); XL = "x: judge prefers A over B (log-odds)"; YL = "y: judge's log-odds of “tie”"; ZL = "z: judge's mean score (1–9)"
fig = plt.figure(figsize=(6.875, 4.75)); gs = fig.add_gridspec(2, 3, width_ratios=[1.12, 1, 1], height_ratios=[1, 1], left=.06, right=.99, top=.945, bottom=.085, wspace=.34, hspace=.5)
# (a) the space: three outputs of the judge for one pair
a = fig.add_subplot(gs[0, 0], projection="3d")
for k in ("A", "B", "other", "good", "fail"):
    m = lab[sub] == k; a.scatter(d[sub][m], t[sub][m], s[sub][m], s=2.5, color=COL[k], alpha=.45, lw=0, depthshade=False, rasterized=True)
a.set_xlabel("x: A over B\n(log-odds)", labelpad=-6, fontsize=6.6, linespacing=1.0); a.set_ylabel("y: tie\n(log-odds)", labelpad=-6, fontsize=6.6, linespacing=1.0); a.set_zlabel("z: mean score (1–9)", labelpad=-10, fontsize=6.6); a.view_init(elev=20, azim=-58); a.tick_params(pad=-4, labelsize=5.5)
a.set_title("(a) The judge's outputs as a space", loc="left", y=1.0, x=-.12)
# (b) x alone
a = fig.add_subplot(gs[0, 1]); bins = np.linspace(-12, 12, 41)
a.hist([d[lab == k] for k in ("A", "B", "other", "good", "fail")], bins=bins, stacked=True, color=[COL[k] for k in ("A", "B", "other", "good", "fail")], lw=0)
a.set_xlabel(XL); a.set_ylabel("number of pairs"); a.set_title("(b) Along the A–B axis (x): ties on both sides", loc="left", x=-.2); a.set_xlim(-12, 12)
a.text(.5, .97, "the judge prefers B  ←      →  prefers A", transform=a.transAxes, ha="center", va="top", fontsize=6, color="#4a4946"); a.set_ylim(0, a.get_ylim()[1] * 1.12)
# (c) x and the tie probability as given (linear)
a = fig.add_subplot(gs[0, 2])
for k in ("A", "B", "other", "good", "fail"):
    m = lab[sub] == k; a.scatter(d[sub][m], Q[sub][m, 2], s=3, color=COL[k], alpha=.5, lw=0, rasterized=True)
a.set_ylim(-.03, 1); a.set_xlim(-12, 12); a.set_xlabel(XL); a.set_ylabel("judge's probability of “tie”"); a.set_title("(c) Same axis, tie probability: ties on the floor", loc="left", x=-.27)
a.axhline(.5, color="#111", lw=.7, ls=(0, (3, 2)))
a.text(.5, .56, "median of the human ties: %.0e\n%.0f%% of them lie below 0.01" % (np.median(Q[istie, 2]), 100 * np.mean(Q[istie, 2] < .01)), transform=a.transAxes, ha="center", va="bottom", fontsize=6.2, color="#111", bbox=dict(fc="white", ec="none", alpha=.8, pad=1.5))
# phase maps
def phase(ax, x, y, xl, yl, title, xr, yr, n=30, tx=0):
    xe = np.linspace(*xr, n + 1); ye = np.linspace(*yr, n + 1); keys = list(COL); H = np.stack([np.histogram2d(x[lab == k], y[lab == k], bins=[xe, ye])[0] for k in keys]); tot = H.sum(0); win = H.argmax(0); pur = H.max(0) / np.maximum(tot, 1)
    img = np.ones((n, n, 4)); from matplotlib.colors import to_rgb
    for ix in range(n):
        for iy in range(n):
            if tot[ix, iy] >= 3:
                img[iy, ix, :3] = to_rgb(COL[keys[win[ix, iy]]]); img[iy, ix, 3] = .25 + .75 * (pur[ix, iy] - 1 / len(keys)) / (1 - 1 / len(keys))
            else:
                img[iy, ix, 3] = 0
    ax.imshow(img, origin="lower", extent=[xr[0], xr[1], yr[0], yr[1]], aspect="auto", interpolation="nearest"); ax.set_xlabel(xl); ax.set_ylabel(yl); ax.set_title(title, loc="left", x=tx)
tr = tuple(np.percentile(t, [.5, 99.5]))
a = fig.add_subplot(gs[1, 1]); phase(a, d, t, XL, YL, "(d) Seen from above (x, y): ties form a band", (-12, 12), (tr[0], max(tr[1], 4.5)), tx=-.12)
a.axhline(0, color="#111", lw=.7, ls=(0, (3, 2))); a.text(11.6, .35, "above this line the judge answers “tie”", ha="right", va="bottom", fontsize=6.0, color="#111", bbox=dict(fc="white", ec="none", alpha=.8, pad=1))
a = fig.add_subplot(gs[1, 2]); phase(a, t, s, YL, ZL, "(e) Seen from the side (y, z): ties score low", tr, (1, 9), tx=-.2)
a = fig.add_subplot(gs[1, 0]); a.axis("off")
share = {k: 100 * np.mean(lab == k) for k in COL}
a.legend(handles=[Line2D([], [], marker="s", ls="", color=COL[k], label=NAME[k] + (" (%.0f%%)" % share[k] if share[k] >= .95 else " (%.1f%%)" % share[k]), markersize=6) for k in COL], loc="upper left", bbox_to_anchor=(-.12, 1.0), title="Colour: what people said about the pair", title_fontsize=7, handletextpad=.3, alignment="left")
a.text(-.12, .0, "Axes: outputs of the judge only.\nx  “Which is better?”, mean of the two orders\ny  “…or a tie?”: tie against the stronger of A, B\nz  “How good is each?” (1–9), mean of the two\n(d), (e): each cell has the colour of its most\ncommon human label; stronger = purer.", transform=a.transAxes, fontsize=6.3, color="#4a4946", va="bottom", linespacing=1.3)
tag = SPEC.replace("/", "_"); fig.savefig(f"{OUT}/F3_phase_{tag}.pdf", dpi=300); fig.savefig(f"{OUT}/F3_phase_{tag}.png", dpi=220)
auc = lambda v, m: __import__("sklearn.metrics", fromlist=["x"]).roc_auc_score(m, v)
print("shares", {k: round(v, 2) for k, v in share.items()}, "| ties below .01: %.3f, median %.2e | AUROC for human tie: |x| %.3f (low |x| = tie), y %.3f, -z %.3f | ties with x<0: %.3f | y>0 among ties %.3f" % (np.mean(Q[istie, 2] < .01), np.median(Q[istie, 2]), auc(-np.abs(d), istie), auc(t, istie), auc(-s, istie), np.mean(d[istie] < 0), np.mean(t[istie] > 0)))
plt.show()

## Cross-model figure (Sec. 4.1) — light
Reads `data/cross_model.json`; also rewrites the rows of the supplement table (`TABLE_ROWS`).

In [ ]:
fresh()
"""Figure F7 and table tab_cross_model (2026-11-15): the four facts of the motivating experiment, judge by judge
(six open VLMs and GPT-4o on ImagenHub edit in the figure; all three datasets in the table).  Data: data/cross_model.json
(agent_reward_bench/cross_model.py).  usage: python fig_cross_model.py <fig_dir> <table_rows.tex>"""
import os, sys, json
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from figstyle import C, setup, DATA
setup(); TEN = TABLE_ROWS
R = json.load(open(os.path.join(DATA, "cross_model.json")))
NAME = {"qwen3vl8b": "Qwen3-VL-8B", "qwen3vl32b": "Qwen3-VL-32B", "internvl35_8b": "InternVL3.5-8B", "internvl35_38b": "InternVL3.5-38B", "qwen35_9b": "Qwen3.5-9B", "qwen25vl7b": "Qwen2.5-VL-7B", "gpt4o": "GPT-4o (closed)"}
X = [r for r in R if r["data"] == "imagenhub/edit"]; n = len(X); yy = np.arange(n)[::-1]
fig, ax = plt.subplots(1, 4, figsize=(6.875, 1.74), sharey=True, gridspec_kw=dict(wspace=.10, left=.145, right=.99, top=.775, bottom=.245))
def base(a, title, xl):
    a.set_title(title, fontsize=7, loc="left", pad=14); a.set_xlabel(xl, fontsize=6.6, labelpad=2); a.tick_params(labelsize=6.2, length=2, pad=1.5); a.grid(axis="y", visible=False); a.set_ylim(-.6, n - .4)
    a.axhspan(-.5, .5, color=C["grid"], alpha=.45, zorder=0)
def leg(a, items):
    for k, (lab, col, mk, fill) in enumerate(items):
        a.plot([], [], mk, color=col, mfc=col if fill else "white", ms=4, label=lab)
    a.legend(loc="lower left", bbox_to_anchor=(-.03, .985), ncol=len(items), fontsize=5.8, handletextpad=.1, columnspacing=.7, borderaxespad=0)
a = ax[0]; base(a, "(a) Forced choice flips\nwith the order", "share of pairs")
for y, r in zip(yy, X):
    a.plot([r["flip_dec"], r["flip_tie"]], [y, y], color=C["axis"], lw=1.2, zorder=1); a.plot(r["flip_dec"], y, "o", color=C["baseline"], ms=4); a.plot(r["flip_tie"], y, "o", color=C["fail"], ms=4)
a.set_xlim(0, .55); a.set_yticks(yy); a.set_yticklabels([NAME[r["judge"]] for r in X], fontsize=6.4); leg(a, [("decided", C["baseline"], "o", True), ("tied", C["fail"], "o", True)])
a = ax[1]; base(a, "(b) Answers “tie” on\npairs people call ties", "share of tied pairs")
for y, r in zip(yy, X):
    a.plot([r["says_tie_on_ties"], r["tie_top_after_ties"]], [y, y], color=C["axis"], lw=1.2, zorder=1); a.plot(r["says_tie_on_ties"], y, "o", color=C["ours"], mfc="white", ms=4); a.plot(r["tie_top_after_ties"], y, "o", color=C["ours"], ms=4)
a.set_xlim(-.03, 1); a.set_xticks([0, .5, 1]); leg(a, [("as given", C["ours"], "o", False), ("one weight per answer", C["ours"], "o", True)])
a = ax[2]; base(a, "(c) Tie probability:\ntied vs. decided pairs", "AUROC")
for y, r in zip(yy, X): a.plot(r["auc"], y, "o", color=C["ours"], ms=4)
a.axvline(.5, color=C["muted"], lw=.8, ls=(0, (3, 2))); a.text(.505, n - .55, "chance", fontsize=5.6, color=C["muted"], va="top"); a.set_xlim(.45, .9)
a = ax[3]; base(a, "(d) Median tie\nprobability, tied pairs", "probability (log scale)")
for y, r in zip(yy, X): a.plot(r["median_pT_ties"], y, "o", color=C["fail"], ms=4)
a.set_xscale("log"); a.set_xlim(3e-5, 1); a.axvline(1 / 3, color=C["muted"], lw=.8, ls=(0, (3, 2))); a.text(.27, n - .55, "can be\nthe answer", fontsize=5.6, color=C["muted"], va="top", ha="right")
fig.savefig(f"{OUT}/F7_cross_model.pdf", bbox_inches=None); fig.savefig(f"{OUT}/F7_cross_model.png", dpi=240, bbox_inches=None)
f2 = lambda v: ("%.2f" % v).lstrip("0") if v < 1 else "1.00"
# rows only, language-neutral: the dataset names and the 'closed' tag are macros defined where the table is used
L = []
for d, mac in (("imagenhub/edit", "\\cmIH"), ("genai_arena/edition", "\\cmGA"), ("agent", "\\cmAG")):
    for k, r in enumerate([r for r in R if r["data"] == d]):
        nm = "GPT-4o \\cmClosed" if r["judge"] == "gpt4o" else NAME[r["judge"]]
        L.append(f"{mac if k == 0 else ''} & {nm} & {f2(r['flip_tie'])} & {f2(r['flip_dec'])} & {f2(r['says_tie_on_ties'])} & {f2(r['tie_top_after_ties'])} & {f2(r['tie_top_after_dec'])} & {f2(r['auc'])} \\\\")
    L.append("\\midrule" if d != "agent" else "\\bottomrule")
open(TEN, "w").write("% generated by paper_figures_nb/fig_cross_model.py from data/cross_model.json; do not edit by hand\n" + "\n".join(L) + "\n")
op = [r for r in R if r["judge"] != "gpt4o"]
for d in ("imagenhub/edit", "genai_arena/edition", "agent"):
    S = [r for r in R if r["data"] == d]; g = lambda k: (min(r[k] for r in S), max(r[k] for r in S)); o5 = [r for r in S if r["judge"] in ("qwen3vl8b", "internvl35_8b", "qwen35_9b", "qwen25vl7b", "qwen3vl32b")]
    print(d, len(S), "judges | flip tie %.2f-%.2f dec %.2f-%.2f | ratio min %.2f | says %.3f-%.3f (mean of the five judges of motivation_exp: %.3f) | auc %.2f-%.2f | after: ties %.2f-%.2f dec %.2f-%.2f" %
          (*g("flip_tie"), *g("flip_dec"), min(r["flip_tie"] / r["flip_dec"] for r in S), *g("says_tie_on_ties"), np.mean([r["says_tie_on_ties"] for r in o5]) if o5 else -1, *g("auc"), *g("tie_top_after_ties"), *g("tie_top_after_dec")))
plt.show()

## Failure kinds, single-column version (Sec. 4.3)
Five judges; pairs chosen by a fixed rule.

In [ ]:
fresh()
"""Figure F5 (2026-11-14): two visual kinds of shared failure and what five judges make of them.  ImagenHub edit pairs with
both human ratings 0.  'No-op': both edits within the lowest fifth of change from the source; 'destructive': both within the
highest fifth (change = mean absolute RGB difference from the source at 128 x 128, fifths over failed candidates).
Examples: hashed order, distinct sources, first three per group (no hand-picking).  Shown per pair: geometric mean over five
judges of the tie probability, how many of the five judges flip with the order, mean pointwise scores.
usage (repo root): python fig_failure_kinds.py <out_dir>"""
import sys, ast, zlib, json
import numpy as np
from PIL import Image
import matplotlib
import matplotlib.pyplot as plt
sys.path.insert(0, "/media/shawn/C/papers/EAGLE-VLM"); sys.path.insert(0, "/media/shawn/C/papers/EAGLE-VLM/scripts")
import absolute_quad as AQ
SPEC = "imagenhub/edit"; ROOT = "/media/shawn/C/papers/VLM/judgebench_local/"; lit = lambda v: ast.literal_eval(v) if isinstance(v, str) else v
JUD = ["qwen3vl8b", "internvl35_8b", "qwen35_9b", "qwen25vl7b", "qwen3vl32b"]
cfg = AQ.load_cfg(f"configs/exp/{AQ.SPECS[SPEC]}.yaml"); P_ = cfg["paths"]; st = AQ.Store(P_["items"], P_["outputs"], P_.get("items_override"), P_.get("snapshot")); ds, I = AQ.dataset_items(st, SPEC)
c = {}
def load(p):
    if p not in c: c[p] = np.asarray(Image.open(ROOT + p).convert("RGB").resize((128, 128), Image.BILINEAR), float) / 255
    return c[p]
SG = {j: AQ.signals(st, ds, j, I) for j in JUD}; ids = sorted(set.intersection(*[set(v) for v in SG.values()])); PWs = {j: AQ.pointwise_feats(st, ds, j, ids) for j in JUD}
LT = np.mean([[np.log10(max((SG[j][i][2][2] + SG[j][i][3][2]) / 2, 1e-12)) for i in ids] for j in JUD], 0)
FL = np.sum([[(SG[j][i][0] > .5) != (SG[j][i][1] > .5) for i in ids] for j in JUD], 0); SC = np.nanmean([PWs[j][:, :2] for j in JUD], 0)
R = np.array([AQ.ratings(SPEC, I[i]) for i in ids], float); CH = np.array([[np.abs(load(lit(I[i]["context"])[0]) - load(x)).mean() for x in lit(I[i]["cand"])] for i in ids])
q20, q80 = np.quantile(CH[R < .5], [.2, .8]); z = R.max(1) == 0; noop = z & (CH < q20).all(1); destr = z & (CH > q80).all(1)
for nm, m in (("no-op", noop), ("destructive", destr), ("other both-zero", z & ~noop & ~destr)):
    print("%-16s n %4d | tie prob (geometric mean over judges; median over pairs) %.1e | judges that flip (mean of 5) %.2f | both mean scores < 5: %.2f | mean score %.2f" % (nm, m.sum(), 10 ** np.median(LT[m]), FL[m].mean(), (SC[m].max(1) < 5).mean(), SC[m].mean()))
order = sorted(range(len(ids)), key=lambda k: zlib.crc32(ids[k].encode()))
def pick(m):
    out, seen = [], set()
    for k in order:
        s = lit(I[ids[k]]["context"])[0]
        if m[k] and s not in seen: out.append(k); seen.add(s)
        if len(out) == 3: break
    return out
G = [("Both edits failed by changing almost nothing", pick(noop), "#1f6fd0"), ("Both edits failed by destroying the image", pick(destr), "#d9541e")]
plt.rcParams.update({"font.family": "serif", "font.serif": ["STIXGeneral", "DejaVu Serif"], "font.size": 7.2, "pdf.fonttype": 42})
def sq(p, side=260):
    im = Image.open(ROOT + p).convert("RGB"); w, h = im.size; s = min(w, h); return im.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s)).resize((side, side), Image.LANCZOS)
fig = plt.figure(figsize=(3.35, 2.5)); outer = fig.add_gridspec(2, 1, left=.006, right=.994, top=.935, bottom=.07, hspace=.34)
SHORT = ["Both failed by changing almost nothing", "Both failed by destroying the image"]
for g, (title, ks, col) in enumerate(G):
    k = ks[0]; sub = outer[g].subgridspec(1, 3, wspace=.03); x = I[ids[k]]; ctx = lit(x["context"]); cd = lit(x["cand"])
    for cc, (p, nm) in enumerate(((ctx[0], "source"), (cd[0], "edit A"), (cd[1], "edit B"))):
        ax = fig.add_subplot(sub[0, cc]); ax.imshow(sq(p)); ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values(): sp.set_linewidth(.4); sp.set_color("#bbbbbb")
        ax.text(.03, .97, nm, transform=ax.transAxes, va="top", fontsize=5.6, color="white", bbox=dict(boxstyle="round,pad=.12", fc=(0, 0, 0, .55), ec="none"))
        if cc == 0:
            ins = x["question"].strip(); ax.set_title(SHORT[g] + f"  (people rate them {R[k, 0]:g} and {R[k, 1]:g})", fontsize=6.3, loc="left", pad=2, color=col, fontweight="bold")
            ax.text(.03, .04, "\u201c" + (ins if len(ins) <= 34 else ins[:32] + "\u2026") + "\u201d", transform=ax.transAxes, va="bottom", fontsize=5.0, style="italic", color="white", bbox=dict(boxstyle="round,pad=.12", fc=(0, 0, 0, .55), ec="none"))
        if cc == 1:
            e = int(np.floor(LT[k])); mant = 10 ** (LT[k] - e)
            ax.set_xlabel(f"five judges:  tie probability ${mant:.0f}\\times10^{{{e}}}$    choice flips for {int(FL[k])} of 5    scores {SC[k, 0]:.1f}, {SC[k, 1]:.1f}", fontsize=6.0, labelpad=2, color="#111")
fig.savefig(f"{OUT}/F5s_failure_kinds_small.pdf", dpi=300); fig.savefig(f"{OUT}/F5s_failure_kinds_small.png", dpi=240)
print("examples:", [[ids[k] for k in ks] for _, ks, _ in G], [[round(float(LT[k]), 2) for k in ks] for _, ks, _ in G])
plt.show()

## Failure kinds, full version (supplement)

In [ ]:
fresh()
"""Figure F5 (2026-11-14): two visual kinds of shared failure and what five judges make of them.  ImagenHub edit pairs with
both human ratings 0.  'No-op': both edits within the lowest fifth of change from the source; 'destructive': both within the
highest fifth (change = mean absolute RGB difference from the source at 128 x 128, fifths over failed candidates).
Examples: hashed order, distinct sources, first three per group (no hand-picking).  Shown per pair: geometric mean over five
judges of the tie probability, how many of the five judges flip with the order, mean pointwise scores.
usage (repo root): python fig_failure_kinds.py <out_dir>"""
import sys, ast, zlib, json
import numpy as np
from PIL import Image
import matplotlib
import matplotlib.pyplot as plt
sys.path.insert(0, "/media/shawn/C/papers/EAGLE-VLM"); sys.path.insert(0, "/media/shawn/C/papers/EAGLE-VLM/scripts")
import absolute_quad as AQ
SPEC = "imagenhub/edit"; ROOT = "/media/shawn/C/papers/VLM/judgebench_local/"; lit = lambda v: ast.literal_eval(v) if isinstance(v, str) else v
JUD = ["qwen3vl8b", "internvl35_8b", "qwen35_9b", "qwen25vl7b", "qwen3vl32b"]
cfg = AQ.load_cfg(f"configs/exp/{AQ.SPECS[SPEC]}.yaml"); P_ = cfg["paths"]; st = AQ.Store(P_["items"], P_["outputs"], P_.get("items_override"), P_.get("snapshot")); ds, I = AQ.dataset_items(st, SPEC)
c = {}
def load(p):
    if p not in c: c[p] = np.asarray(Image.open(ROOT + p).convert("RGB").resize((128, 128), Image.BILINEAR), float) / 255
    return c[p]
SG = {j: AQ.signals(st, ds, j, I) for j in JUD}; ids = sorted(set.intersection(*[set(v) for v in SG.values()])); PWs = {j: AQ.pointwise_feats(st, ds, j, ids) for j in JUD}
LT = np.mean([[np.log10(max((SG[j][i][2][2] + SG[j][i][3][2]) / 2, 1e-12)) for i in ids] for j in JUD], 0)
FL = np.sum([[(SG[j][i][0] > .5) != (SG[j][i][1] > .5) for i in ids] for j in JUD], 0); SC = np.nanmean([PWs[j][:, :2] for j in JUD], 0)
R = np.array([AQ.ratings(SPEC, I[i]) for i in ids], float); CH = np.array([[np.abs(load(lit(I[i]["context"])[0]) - load(x)).mean() for x in lit(I[i]["cand"])] for i in ids])
q20, q80 = np.quantile(CH[R < .5], [.2, .8]); z = R.max(1) == 0; noop = z & (CH < q20).all(1); destr = z & (CH > q80).all(1)
for nm, m in (("no-op", noop), ("destructive", destr), ("other both-zero", z & ~noop & ~destr)):
    print("%-16s n %4d | tie prob (geometric mean over judges; median over pairs) %.1e | judges that flip (mean of 5) %.2f | both mean scores < 5: %.2f | mean score %.2f" % (nm, m.sum(), 10 ** np.median(LT[m]), FL[m].mean(), (SC[m].max(1) < 5).mean(), SC[m].mean()))
order = sorted(range(len(ids)), key=lambda k: zlib.crc32(ids[k].encode()))
def pick(m):
    out, seen = [], set()
    for k in order:
        s = lit(I[ids[k]]["context"])[0]
        if m[k] and s not in seen: out.append(k); seen.add(s)
        if len(out) == 3: break
    return out
G = [("Both edits failed by changing almost nothing", pick(noop), "#1f6fd0"), ("Both edits failed by destroying the image", pick(destr), "#d9541e")]
plt.rcParams.update({"font.family": "serif", "font.serif": ["STIXGeneral", "DejaVu Serif"], "font.size": 7.2, "pdf.fonttype": 42})
def sq(p, side=260):
    im = Image.open(ROOT + p).convert("RGB"); w, h = im.size; s = min(w, h); return im.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s)).resize((side, side), Image.LANCZOS)
fig = plt.figure(figsize=(6.875, 3.85)); outer = fig.add_gridspec(1, 2, left=.004, right=.996, top=.925, bottom=.035, wspace=.045)
for g, (title, ks, col) in enumerate(G):
    sub = outer[g].subgridspec(3, 3, wspace=.03, hspace=.40); fig.text(.25 + .5 * g, .972, title, ha="center", va="center", fontsize=8.2, fontweight="bold", color=col)
    for r, k in enumerate(ks):
        x = I[ids[k]]; ctx = lit(x["context"]); cd = lit(x["cand"])
        for cc, (p, nm) in enumerate(((ctx[0], "source"), (cd[0], "edit A"), (cd[1], "edit B"))):
            ax = fig.add_subplot(sub[r, cc]); ax.imshow(sq(p)); ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values(): sp.set_linewidth(.4); sp.set_color("#bbbbbb")
            ax.text(.03, .97, nm, transform=ax.transAxes, va="top", fontsize=5.8, color="white", bbox=dict(boxstyle="round,pad=.12", fc=(0, 0, 0, .55), ec="none"))
            if cc == 0:
                ins = x["question"].strip(); ax.set_title("“" + (ins if len(ins) <= 40 else ins[:38] + "…") + "”", fontsize=6.6, loc="left", pad=1.5, style="italic")
            if cc == 1:
                e = int(np.floor(LT[k])); mant = 10 ** (LT[k] - e)
                ax.set_xlabel(f"tie probability ${mant:.0f}\\times10^{{{e}}}$     order flips: {int(FL[k])} of 5 judges     scores {SC[k, 0]:.1f} and {SC[k, 1]:.1f}", fontsize=6.0, labelpad=1.5, color="#111")
fig.savefig(f"{OUT}/F5_failure_kinds.pdf", dpi=300); fig.savefig(f"{OUT}/F5_failure_kinds.png", dpi=220)
print("examples:", [[ids[k] for k in ks] for _, ks, _ in G], [[round(float(LT[k]), 2) for k in ks] for _, ks, _ in G])
plt.show()

## Architecture figure — light

In [ ]:
fresh()
"""Architecture figure for paper_v2_dropped_ties (2026-11-14): which modules are standard and shared with the baselines,
which are this paper's, and why the fitted state head is needed.  Numbers: EXPERIMENT_RESULTS_20261109.md, sections
"只换问法的认证对比" and "不拟合、不认证时各问法直接做决定" (ImagenHub edit, A/B/tie question: 3.1 -> 34.9 correct rejections).
run: ../.figenv/bin/python fig_architecture.py"""
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from figstyle import C, setup, save
setup()
fig, ax = plt.subplots(figsize=(7.1, 3.25)); ax.set_xlim(0, 100); ax.set_ylim(0, 46); ax.axis("off")
X = [1, 21.5, 42, 62.5, 83]; W = 16; GREY, OURS = C["baseline"], C["ours"]


def box(x, y, h, title, lines, edge, fill="white", dash=False, tcol=None):
    ax.add_patch(FancyBboxPatch((x, y), W, h, boxstyle="round,pad=0.25,rounding_size=1.2", fc=fill, ec=edge, lw=1.1, ls=(0, (4, 2)) if dash else "-"))
    ax.text(x + W / 2, y + h - 1.6, title, ha="center", va="top", fontsize=7.2, fontweight="bold", color=tcol or C["ink"])
    ax.text(x + W / 2, y + h - 5.0, "\n".join(lines), ha="center", va="top", fontsize=6.3, color=C["ink2"], linespacing=1.35)


def arrow(x0, x1, y, col=C["muted"]):
    ax.annotate("", xy=(x1, y), xytext=(x0, y), arrowprops=dict(arrowstyle="-|>", color=col, lw=1.0, shrinkA=0, shrinkB=0))


# column headers
for x, t in zip(X, ["1  Question", "2  Signal", "3  State head", "4  What may be said", "5  Certificate"]):
    ax.text(x + W / 2, 45.2, t, ha="center", va="center", fontsize=7.2, color=C["muted"])
# row labels
ax.text(0.2, 40.6, "Winner-only certified judge", fontsize=6.6, color=C["muted"], va="center", style="italic")
ax.text(0.2, 25.2, "With ties kept (this paper)", fontsize=6.6, color=OURS, va="center", style="italic")
# top row: winner-only
yt, ht = 30.0, 9.0
box(X[0], yt, ht, "A or B?", ["both orders"], GREY)
box(X[1], yt, ht, "confidence", ["of the choice"], GREY)
ax.text(X[2] + W / 2, yt + ht / 2, "none: one statement,\nraw confidence\nis enough", ha="center", va="center", fontsize=6.3, color=C["muted"])
box(X[3], yt, ht, "“A is better”", ["or abstain"], GREY)
box(X[4], yt, ht, "threshold", ["on labelled", "calibration pairs"], GREY)
for i in range(4):
    arrow(X[i] + W + .6, X[i + 1] - .6, yt + ht / 2)
# bottom row: ours
yb, hb = 6.5, 17.0
box(X[0], yb, hb, "three questions", ["A or B?", "A, B or tie?", "how good is each?", "(1 to 9)"], OURS, tcol=OURS)
box(X[1], yb, hb, "direction,", ["tie probability,", "level of each", "candidate"], C["axis"])
box(X[2], yb, hb, "fitted head", ["on labelled pairs:", "P(both acceptable)", "P(only A), P(only B)", "P(neither)"], OURS, dash=True)
box(X[3], yb, hb, "more outputs", ["ship A, ship B,", "reject both, escalate", "(relative: better,", "tie, not worse)"], OURS, tcol=OURS)
box(X[4], yb, hb, "threshold", ["on labelled", "calibration pairs", "(Learn-then-Test)"], GREY)
for i in range(4):
    arrow(X[i] + W + .6, X[i + 1] - .6, yb + hb / 2, OURS if i in (1, 2) else C["muted"])
# footnotes
ax.text(X[0] + W / 2, 2.6, "this paper:\nwhat to ask", ha="center", va="center", fontsize=6.3, color=OURS)
ax.text(X[2] + W / 2, 2.6, "required by 4;\na standard technique", ha="center", va="center", fontsize=6.3, color=OURS)
ax.text(X[3] + W / 2 + 6, 2.6, "", fontsize=6)
ax.text(X[4] + W / 2, 2.6, "standard; shared\nwith the baselines", ha="center", va="center", fontsize=6.3, color=C["muted"])
ax.text(X[3] + W / 2, 2.6, "this paper:\nwhat may be said", ha="center", va="center", fontsize=6.3, color=OURS)
save(fig, "A1_architecture")
plt.show()

## Window figure (Sec. 6) — light
Reads `data/theory_fit.json` and the three `shared_failure_curve*.csv` tables.

In [ ]:
fresh()
"""Single-column versions of two figures for the 8-page main text (2026-11-15).
T1s_window_small: the window figure (same data and model as T1_interval_theory), three narrow panels.
usage: python fig_small.py <out_dir>"""
import os, sys, json
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from figstyle import C, setup, DATA
setup()
TH = json.load(open(os.path.join(DATA, "theory_fit_v2.json")))      # corrected simulation (start of the test sequence chosen on an independent fit sample); the first version is theory_fit.json
SHOW_AGENT = False     # editable: draw the four web-agent source benchmarks as diamonds (removed from the paper figure, 2026-11-15)
AGW = json.load(open(os.path.join(DATA, "agent_window.json")))     # web-agent source benchmarks at the three targets (four judges)
OBS = {".1": "shared_failure_curve_alpha10.csv", ".2": "shared_failure_curve.csv", ".3": "shared_failure_curve_alpha30.csv"}
fig, axes = plt.subplots(1, 3, figsize=(3.35, 1.75), sharey=True, gridspec_kw=dict(wspace=.07, left=.13, right=.995, top=.86, bottom=.3))
for ax, (a, ttl) in zip(axes, ((".1", r"$\alpha=.1$ (not fitted)"), (".2", r"$\alpha=.2$ (fitted)"), (".3", r"$\alpha=.3$ (not fitted)"))):
    key = "0" + a; cv = TH["open"]["curves"][key]; hi = TH["pi_hi"][key]
    ax.axvspan(hi, 1, color=C["grid"], alpha=.6, zorder=0)
    g = np.convolve(np.array(cv["tie"]) - np.array(cv["base"]), np.ones(5) / 5, mode="same")
    ax.plot(cv["pi"][2:-2], g[2:-2], color=C["ink2"], lw=1.2, label="model")
    d = pd.read_csv(os.path.join(DATA, OBS[a])); d = d[(d.scope == "open") & (d.tau == "mid")].sort_values("pi")
    ax.errorbar(d.pi, d.gain_vs_PW, yerr=[d.gain_vs_PW - d.gain_vs_PW_lo, d.gain_vs_PW_hi - d.gain_vs_PW], fmt="o", color=C["ours"], ms=3.2, mec="white", mew=.5, elinewidth=.8, label="measured")
    # trend of the measured points: local-linear smoother (Gaussian kernel, bandwidth .07 in pi); a guide for the eye, not a model
    xs = np.linspace(d.pi.min(), d.pi.max(), 120); X, Y = d.pi.values, d.gain_vs_PW.values; sm = []
    for x0 in xs:
        w = np.exp(-.5 * ((X - x0) / .07) ** 2); A = np.c_[np.ones_like(X), X - x0] * np.sqrt(w)[:, None]; sm.append(np.linalg.lstsq(A, Y * np.sqrt(w), rcond=None)[0][0])
    ax.plot(xs, sm, color=C["ours"], lw=1.0, ls=(0, (3, 1.5)), alpha=.9, label="measured, smoothed", zorder=2)
    print(a, "smoothed peak at pi = %.2f (%.1f)" % (xs[int(np.argmax(sm))], max(sm)), "| model peak at %.2f" % cv["pi"][2:-2][int(np.argmax(g[2:-2]))])
    ax.axhline(0, color=C["axis"], lw=.8); ax.set_xlim(.3, 1.0); ax.set_xticks([.4, .6, .8]); ax.set_ylim(-3, 10); ax.set_yticks([0, 4, 8])
    ax.set_title(ttl, fontsize=6.6, fontweight="normal", loc="center", pad=2); ax.tick_params(labelsize=6, length=2, pad=1.5)
# other data at alpha = .2: the four source benchmarks of the web-agent pairs (four judges; EXPERIMENT_RESULTS, addendum 20): share of shared failure, gain, 95% interval
for ax, a in zip(axes, ("0.1", "0.2", "0.3")):
    for b in ("webarena", "visualwebarena", "workarena", "assistantbench"):
        if not SHOW_AGENT: continue
        v = AGW[f"{b}@{a}"]; ax.plot([v["pi"], v["pi"]], [v["lo"], v["hi"]], color=C["fail"], lw=.8, zorder=3); ax.plot(v["pi"], v["gain"], "D", color=C["fail"], ms=3.0, mec="white", mew=.5, zorder=4)
if SHOW_AGENT: axes[2].plot([], [], "D", color=C["fail"], ms=3.0, mec="white", mew=.5, ls="", label="web-agent benchmarks")
axes[1].set_xlabel("share of pairs in which both candidates failed", fontsize=6.6, labelpad=2)
axes[0].set_ylabel("decisions added by the\ntie probability (per 100)", fontsize=6.4, labelpad=2)
axes[2].legend(loc="upper left", fontsize=5.4, handlelength=1.2, handletextpad=.3, borderaxespad=.1, labelspacing=.2)
fig.savefig(f"{OUT}/T1s_window_small.pdf", bbox_inches=None); fig.savefig(f"{OUT}/T1s_window_small.png", dpi=260, bbox_inches=None)
plt.show()

## Attention probe, registers (supplement)

In [ ]:
fresh()
"""Analysis of the second attention probe (attn_probe2.py): sink tokens, where they come from, and attention relative to a
content-free prompt.  Qwen3-VL-8B, forced-choice prompt, ImagenHub edit pairs, both orders.
 sink tokens: image tokens with a massive activation in the dimensions in which the first token of the sequence has one
              (Kang et al., ICLR 2025): score = max_d |x_d| / RMS(x) at the middle layer; threshold = 20 unless stated.
 relative attention (Zhang et al., ICLR 2025): attention under the judge prompt divided by attention under 'Describe the
              images.' for the same images and order, per token and layer.
 pointing score: share of an edit's attention that falls on the 20% of its cells that the edit changed most (chance .20).
usage: python attn2_fig.py <probe_dir> <out_dir>"""
import sys, json, ast, collections
import numpy as np
from PIL import Image
import matplotlib
import matplotlib.pyplot as plt
D = ATTN2_DIR; ROOT = "/media/shawn/C/papers/VLM/judgebench_local/"; lit = lambda v: ast.literal_eval(v) if isinstance(v, str) else v
R = collections.defaultdict(dict)
for l in open(f"{D}/probe2.jsonl"):
    r = json.loads(l); R[(r["id"], r["order"])][r["kind"]] = r
rows = {}
ids = {k[0] for k in R}
for l in open(ROOT + "items/imagenhub.jsonl"):
    x = json.loads(l)
    if x["id"] in ids: rows[x["id"]] = x
def change(src, ed, shape):
    h, w = shape; f = lambda p: np.asarray(Image.open(ROOT + p).convert("RGB").resize((w * 8, h * 8), Image.BILINEAR), float) / 255
    return np.abs(f(src) - f(ed)).mean(2).reshape(h, 8, w, 8).mean((1, 3))
def split(v, grid):
    out, k = [], 0
    for t, h, w in grid:
        n = (h // 2) * (w // 2); out.append(v[..., k:k + n].reshape(v.shape[:-1] + (h // 2, w // 2))); k += n
    return out
TH = 20.0; S = collections.defaultdict(list); PL = collections.defaultdict(list); keep = {}; allscore = []
for (i, o), r in R.items():
    if "question" not in r or "generic" not in r: continue
    Zq, Zg = np.load(f"{D}/{r['question']['file']}"), np.load(f"{D}/{r['generic']['file']}")
    if Zq["A"].shape != Zg["A"].shape: continue
    A, G, grid = Zq["A"], Zg["A"], Zq["grid"]; L = A.shape[0]; sc = Zq["sink_score"]; allscore.append(sc); sink = sc >= TH; n0 = Zq["N0"]; hi = n0 > n0.mean() + 3 * n0.std()
    g = "tied" if r["question"]["label"] == "tie" else "decided"; up = slice(L // 2, L)
    S[g].append(dict(n_sink=sink.sum(), sink_frac=sink.mean(), sink_share_up=A[up][:, sink].sum() / A[up].sum(), sink_share_generic=G[up][:, sink].sum() / G[up].sum(),
                     img_share_up=A[up].sum(1).mean(), img_share_generic=G[up].sum(1).mean(), n_hi=hi.sum(), hi_in_sink=(hi & sink).sum() / max(hi.sum(), 1), sink_in_hi=(hi & sink).sum() / max(sink.sum(), 1),
                     norm_ratio=n0[sink].mean() / n0[~sink].mean() if sink.any() else np.nan, corr_qg=np.corrcoef(A[up].mean(0), G[up].mean(0))[0, 1],
                     sink_same_generic=(sink == (Zg["sink_score"] >= TH)).mean()))
    x = rows[i]; ctx = lit(x["context"]); cd = lit(x["cand"]); eds = cd if o == "AB" else cd[::-1]
    Am, Gm, Sm = split(A, grid), split(G, grid), split(sink, grid)
    for q in (1, 2):
        ch = change(ctx[0], eds[q - 1], Am[q].shape[1:]); top = (ch >= np.quantile(ch, .8)).ravel()
        a = Am[q].reshape(L, -1); gq = Gm[q].reshape(L, -1); sk = Sm[q].ravel()
        raw = a[:, top].sum(1) / a.sum(1); am = a * ~sk; masked = am[:, top].sum(1) / np.maximum(am.sum(1), 1e-12)
        rel = a / (gq + 1e-6 * gq.mean(1, keepdims=True) + 1e-12); rel = rel * ~sk; relp = rel[:, top].sum(1) / np.maximum(rel.sum(1), 1e-12)
        PL[g].append(np.stack([raw, masked, relp]))
    keep[(i, o)] = (Am, Gm, Sm, eds, ctx[0])
allscore = np.concatenate(allscore); print("sink score: median %.1f, 99th pct %.1f, share >= 20: %.4f, share >= 10: %.4f, share >= 50: %.4f" % (np.median(allscore), np.percentile(allscore, 99), (allscore >= 20).mean(), (allscore >= 10).mean(), (allscore >= 50).mean()))
for g, V in S.items():
    m = lambda k: float(np.nanmean([v[k] for v in V]))
    print("%-8s calls %3d | sink tokens %.1f per call (%.3f of image tokens) hold %.3f of image attention under the judge prompt and %.3f under the generic prompt | images get %.3f / %.3f of all attention | "
          "sinks are the same tokens under both prompts: %.3f | high-norm input tokens: %.1f per call, %.2f of them are sinks, %.2f of sinks are high-norm, sink / non-sink input norm %.2f | map correlation judge vs generic %.2f" % (
          g, len(V), m("n_sink"), m("sink_frac"), m("sink_share_up"), m("sink_share_generic"), m("img_share_up"), m("img_share_generic"), m("sink_same_generic"), m("n_hi"), m("hi_in_sink"), m("sink_in_hi"), m("norm_ratio"), m("corr_qg")))
P = {g: np.mean(v, 0) for g, v in PL.items()}; Pall = np.mean([x for v in PL.values() for x in v], 0); L = Pall.shape[1]
best = np.argsort(-Pall[2])[:4]; print("pointing score by layer (all pairs): raw max %.3f at layer %d; sink-masked max %.3f at %d; relative max %.3f at %d | best four layers for relative attention: %s | upper-half mean: raw %.3f, masked %.3f, relative %.3f" % (
    Pall[0].max(), Pall[0].argmax(), Pall[1].max(), Pall[1].argmax(), Pall[2].max(), Pall[2].argmax(), sorted(best.tolist()), Pall[0][L // 2:].mean(), Pall[1][L // 2:].mean(), Pall[2][L // 2:].mean()))
for g in P: print("  %s: relative-attention pointing at the best four layers %.3f; raw %.3f" % (g, P[g][2][best].mean(), P[g][0][best].mean()))
allp = np.array([x for v in PL.values() for x in v]); rng = np.random.default_rng(0); bb = rng.integers(0, len(allp), (2000, len(allp)))
for nm, row, sl in (("relative, best four layers", 2, best), ("raw, upper half", 0, np.arange(L // 2, L)), ("raw, best four layers", 0, best)):
    v = allp[:, row][:, sl].mean(1); b = v[bb].mean(1); print("pointing %-28s %.3f [%.3f, %.3f] (n = %d edit views)" % (nm, v.mean(), *np.percentile(b, [2.5, 97.5]), len(v)))
json.dump(dict(point=Pall.tolist(), best=best.tolist(), by={g: P[g].tolist() for g in P}), open(f"{OUT}/attn2_summary.json", "w"))
# ---- figure
plt.rcParams.update({"font.family": "serif", "font.serif": ["STIXGeneral", "DejaVu Serif"], "font.size": 7, "pdf.fonttype": 42, "axes.spines.top": False, "axes.spines.right": False})
teaser = [i for i in json.load(open("/media/shawn/C/papers/VLM/paper_figures_nb/figs_v3/F1_pairs.json")) if (i, "AB") in keep][:3]
fig = plt.figure(figsize=(6.875, 1.0 * len(teaser) + .35)); gs = fig.add_gridspec(len(teaser), 9, width_ratios=[1, 1, 1, 1, 1, 1, 1, .95, 2.0], left=.004, right=.992, top=.9, bottom=.13, wspace=.04, hspace=.07)
def show(ax, p, m, cmap, label, shape):
    h, w = shape; ax.imshow(Image.open(ROOT + p).convert("RGB").resize((w * 14, h * 14), Image.LANCZOS))
    if m is not None:
        z = np.clip(m / np.quantile(m, .995), 0, 1); ax.imshow(np.kron(z, np.ones((14, 14))), cmap=cmap, alpha=.62, vmin=0, vmax=1)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values(): sp.set_visible(True); sp.set_linewidth(.4); sp.set_color("#bbbbbb")
    if label: ax.text(.03, .97, label, transform=ax.transAxes, va="top", fontsize=5.6, color="white", bbox=dict(boxstyle="round,pad=.12", fc=(0, 0, 0, .55), ec="none"))
for n, i in enumerate(teaser):
    Am, Gm, Sm, eds, src = keep[(i, "AB")]; ax = fig.add_subplot(gs[n, 0]); show(ax, src, None, None, "source", Am[0].shape[1:])
    if n == 0: ax.set_title("", fontsize=6.4)
    for q, nm in ((1, "edit A"), (2, "edit B")):
        c0 = 1 + (q - 1) * 3; raw = Am[q][L // 2:].mean(0); rel = (Am[q][best] / (Gm[q][best] + 1e-6 * Gm[q][best].mean() + 1e-12)).mean(0) * ~Sm[q]
        for c, (m, cm_, t) in enumerate(((raw, "inferno", "raw"), (rel, "inferno", "relative"), (change(src, eds[q - 1], raw.shape), "viridis", "edit change"))):
            ax = fig.add_subplot(gs[n, c0 + c]); show(ax, eds[q - 1], m, cm_, nm if c == 0 else None, raw.shape)
            if n == 0: ax.set_title(t, fontsize=6.2, pad=2)
ax = fig.add_subplot(gs[:, 8]); xs = np.arange(L)
ax.plot(xs, Pall[0], color="#8a8882", lw=1.2, label="raw"); ax.plot(xs, Pall[1], color="#c9a227", lw=1.2, ls=(0, (4, 2)), label="sinks removed"); ax.plot(xs, Pall[2], color="#1f6fd0", lw=1.5, label="relative, sinks removed")
ax.axhline(.2, color="#111", lw=.6, ls=(0, (2, 2))); ax.text(L - 1, .197, "chance", ha="right", va="top", fontsize=5.8); ax.set_xlabel("layer"); ax.set_ylabel("share of attention\non the edited region", fontsize=6.4); ax.tick_params(labelsize=6)
ax.legend(fontsize=5.8, frameon=False, loc="lower right", handlelength=1.6, borderaxespad=.2); ax.set_title("Does the judge look at the edit?", fontsize=6.6, loc="center", pad=3)
fig.savefig(f"{OUT}/F6_attention_registers.pdf", dpi=300); fig.savefig(f"{OUT}/F6_attention_registers.png", dpi=220)
plt.show()

## Attention probe, sinks (supplement)

In [ ]:
fresh()
"""Descriptive (2026-11-14): are the hot spots of the judge's image attention sink / register tokens, and where does the
rest of the attention go?  Uses the maps saved by attn_probe.py (Qwen3-VL-8B, forced choice, 124 ImagenHub edit pairs, both
orders; last-position attention, upper half of the layers).  No new model calls.
 1  concentration: share of the image attention held by the top 1 / top 5 tokens of a call; tokens above 10x the image's median
 2  persistence: correlation of a map with the map of the SAME image in the other order (source: same slot; an edit: other slot)
 3  after removing the sink tokens (above 10x the median of their image) and renormalising: does the rest follow the edited
    region?  change map = mean absolute RGB difference between edit and source per token cell; reported: Spearman correlation
    with the change map and the share of attention on the 20% most changed cells (chance .20), raw and sink-removed.
 4  figure F6: three teaser pairs; per image the raw map, the sink-removed map and the change map.
usage: python attn_sinks.py <attn_dir> <out_dir>"""
import sys, json, ast, collections, os
import numpy as np
from scipy.stats import spearmanr
from PIL import Image
import matplotlib
import matplotlib.pyplot as plt
D = ATTN_DIR; ROOT = "/media/shawn/C/papers/VLM/judgebench_local/"; lit = lambda v: ast.literal_eval(v) if isinstance(v, str) else v
R = collections.defaultdict(dict)
for l in open(f"{D}/attn.jsonl"):
    r = json.loads(l); R[r["id"]][r["order"]] = r
rows = {}
for l in open(ROOT + "items/imagenhub.jsonl"):
    x = json.loads(l)
    if x["id"] in R: rows[x["id"]] = x
def cellmap(p, shape):
    h, w = shape; return np.asarray(Image.open(ROOT + p).convert("RGB").resize((w * 8, h * 8), Image.BILINEAR), float) / 255
def change(src, ed, shape):
    h, w = shape; d = np.abs(cellmap(src, shape) - cellmap(ed, shape)).mean(2); return d.reshape(h, 8, w, 8).mean((1, 3))
def desink(m, k=10.0):
    med = np.median(m); s = m > k * med; out = m.copy(); out[s] = med; return out, s
G = collections.defaultdict(list); keep = {}
for i, r in R.items():
    if len(r) < 2: continue
    x = rows[i]; ctx = lit(x["context"]); cd = lit(x["cand"]); g = "tied" if r["AB"]["label"] == "tie" else "decided"
    M = {o: [np.load(f"{D}/{r[o]['file']}")[f"m{q}"] for q in range(3)] for o in ("AB", "BA")}
    for o in ("AB", "BA"):
        allv = np.concatenate([m.ravel() for m in M[o]]); srt = np.sort(allv)[::-1]; tot = allv.sum()
        G[g].append(("top1", srt[0] / tot)); G[g].append(("top5", srt[:5].sum() / tot)); G[g].append(("ntok", len(allv)))
        sinks = [desink(m)[1] for m in M[o]]; G[g].append(("sink_share", sum(m[s].sum() for m, s in zip(M[o], sinks)) / tot)); G[g].append(("sink_n", sum(s.sum() for s in sinks)))
        G[g].append(("sink_share_source", M[o][0][sinks[0]].sum() / M[o][0].sum()))
        eds = cd if o == "AB" else cd[::-1]
        for q in (1, 2):
            m = M[o][q]; ch = change(ctx[0], eds[q - 1], m.shape); clean, s = desink(m); top = ch >= np.quantile(ch, .8)
            G[g].append(("rho_raw", spearmanr(m.ravel(), ch.ravel())[0])); G[g].append(("rho_clean", spearmanr(clean.ravel(), ch.ravel())[0]))
            G[g].append(("hit_raw", m[top].sum() / m.sum())); G[g].append(("hit_clean", clean[top].sum() / clean.sum()))
            if ch.std() > 0: G[g].append(("change_conc", ch[top].sum() / ch.sum()))
    # persistence across orders
    if M["AB"][0].shape == M["BA"][0].shape: G[g].append(("persist_source", np.corrcoef(M["AB"][0].ravel(), M["BA"][0].ravel())[0, 1]))
    for c, (qa, qb) in enumerate(((1, 2), (2, 1))):                       # candidate c: slot qa in AB, slot qb in BA
        if M["AB"][qa].shape == M["BA"][qb].shape:
            G[g].append(("persist_edit", np.corrcoef(M["AB"][qa].ravel(), M["BA"][qb].ravel())[0, 1]))
            a, b = desink(M["AB"][qa])[1], desink(M["BA"][qb])[1]; G[g].append(("sink_overlap", (a & b).sum() / max((a | b).sum(), 1)))
            ca, cb = desink(M["AB"][qa])[0], desink(M["BA"][qb])[0]; G[g].append(("persist_edit_clean", np.corrcoef(ca.ravel(), cb.ravel())[0, 1]))
    keep[i] = M
for g, V in G.items():
    d = collections.defaultdict(list)
    for k, v in V: d[k].append(v)
    m = lambda k: float(np.nanmean(d[k]))
    print("%-8s tokens per call %.0f | top-1 token %.3f, top-5 %.3f of image attention | sink tokens (>10x median): %.1f per call holding %.3f (source image alone: %.3f) | same image across orders: source r %.2f, edit r %.2f (sink-removed %.2f), sink overlap %.2f | "
          "edits vs change map: Spearman raw %.3f, sink-removed %.3f; attention on the 20%% most changed cells raw %.3f, sink-removed %.3f (chance .20; the change itself puts %.2f there)" % (
          g, m("ntok"), m("top1"), m("top5"), m("sink_n"), m("sink_share"), m("sink_share_source"), m("persist_source"), m("persist_edit"), m("persist_edit_clean"), m("sink_overlap"), m("rho_raw"), m("rho_clean"), m("hit_raw"), m("hit_clean"), m("change_conc")))
# ---- figure
teaser = [i for i in json.load(open("/media/shawn/C/papers/VLM/paper_figures_nb/figs_v3/F1_pairs.json")) if i in keep][:3]
plt.rcParams.update({"font.family": "serif", "font.serif": ["STIXGeneral", "DejaVu Serif"], "font.size": 7, "pdf.fonttype": 42})
def show(ax, p, m, cmap, label, shape):
    h, w = shape; im = Image.open(ROOT + p).convert("RGB").resize((w * 14, h * 14), Image.LANCZOS); ax.imshow(im)
    if m is not None:
        z = m / m.max(); ax.imshow(np.kron(z, np.ones((14, 14))), cmap=cmap, alpha=.62, vmin=0, vmax=1)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values(): sp.set_linewidth(.4); sp.set_color("#bbbbbb")
    if label: ax.text(.03, .97, label, transform=ax.transAxes, va="top", fontsize=5.6, color="white", bbox=dict(boxstyle="round,pad=.12", fc=(0, 0, 0, .55), ec="none"))
fig, axes = plt.subplots(len(teaser), 7, figsize=(6.875, .98 * len(teaser) + .3), gridspec_kw=dict(wspace=.03, hspace=.06, left=.004, right=.996, top=.9, bottom=.005))
for n, i in enumerate(teaser):
    x = rows[i]; ctx = lit(x["context"]); cd = lit(x["cand"]); M = keep[i]["AB"]
    show(axes[n, 0], ctx[0], None, None, "source", M[0].shape)
    for q, nm in ((1, "edit A"), (2, "edit B")):
        m = M[q]; c0 = 1 + (q - 1) * 3
        show(axes[n, c0], cd[q - 1], m, "inferno", nm, m.shape); show(axes[n, c0 + 1], cd[q - 1], desink(m)[0], "inferno", None, m.shape); show(axes[n, c0 + 2], cd[q - 1], change(ctx[0], cd[q - 1], m.shape), "viridis", None, m.shape)
for c, t in enumerate(["", "raw attention", "sink tokens removed", "what the edit changed", "raw attention", "sink tokens removed", "what the edit changed"]):
    axes[0, c].set_title(t, fontsize=6.6, pad=2)
fig.savefig(f"{OUT}/F6_attention_sinks.pdf", dpi=300); fig.savefig(f"{OUT}/F6_attention_sinks.png", dpi=220)
plt.show()

## Attention probe, first version (not in the paper)

In [ ]:
fresh()
"""Analysis and figure for the attention probe (agent_reward_bench/attn_20261114; Qwen3-VL-8B, forced choice, 124 ImagenHub
edit pairs: 4 teaser pairs, 60 tied pairs with both ratings 0, 60 decided pairs; both orders).  usage: python attn_fig.py <dir> <out_dir>"""
import sys, os, json, ast, collections
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image
D = ATTN_DIR
R = collections.defaultdict(dict)
for l in open(f"{D}/attn.jsonl"):
    r = json.loads(l); R[r["id"]][r["order"]] = r
rows = {}
for l in open("/media/shawn/C/papers/VLM/judgebench_local/items/imagenhub.jsonl"):
    x = json.loads(l)
    if x["id"] in R: rows[x["id"]] = x
teaser = json.load(open("/media/shawn/C/papers/VLM/paper_figures_nb/figs_v3/F1_pairs.json"))
st = collections.defaultdict(list)
for i, r in R.items():
    if len(r) < 2: continue
    ab, ba = r["AB"], r["BA"]; g = "tied" if ab["label"] == "tie" else "decided"
    tot = lambda q: sum(q["share"])
    # shares within the image attention: source, first shown, second shown
    f_ab, s_ab = ab["share"][1] / tot(ab), ab["share"][2] / tot(ab); f_ba, s_ba = ba["share"][1] / tot(ba), ba["share"][2] / tot(ba)
    c0_ab, c1_ab = f_ab, s_ab; c0_ba, c1_ba = s_ba, f_ba                    # candidate 0 is first in AB, second in BA
    st[g].append(dict(img=(tot(ab) + tot(ba)) / 2, src=(ab["share"][0] / tot(ab) + ba["share"][0] / tot(ba)) / 2, first=(f_ab + f_ba) / 2, second=(s_ab + s_ba) / 2,
                      pos_diff_ab=f_ab - s_ab, pos_diff_ba=f_ba - s_ba, cand_diff_ab=c0_ab - c1_ab, cand_diff_ba=c0_ba - c1_ba,
                      pick_first_ab=ab["pA_first"] > .5, pick_first_ba=ba["pA_first"] > .5, flip=(ab["pA_first"] > .5) == (ba["pA_first"] > .5),   # same slot in both orders = a different candidate
                      pick0_ab=ab["pA_first"] > .5, pick0_ba=ba["pA_first"] <= .5, label=ab["label"]))
for g, V in st.items():
    m = lambda k: np.mean([v[k] for v in V]); a = lambda k: np.array([v[k] for v in V])
    cand = np.corrcoef(a("cand_diff_ab"), a("cand_diff_ba"))[0, 1]; pos = np.corrcoef(a("pos_diff_ab"), a("pos_diff_ba"))[0, 1]
    # does the picked candidate get more attention?  (both orders pooled)
    pick_more = np.mean(np.r_[(a("cand_diff_ab") > 0) == a("pick0_ab"), (a("cand_diff_ba") > 0) == a("pick0_ba")])
    print("%-8s n %3d | attention on images %.3f of all | within images: source %.2f, first-shown edit %.2f, second-shown %.2f | picks the first-shown %.2f | verdict follows the slot (flip) %.2f | "
          "candidate attention difference agrees across orders: r %.2f | slot attention difference agrees across orders: r %.2f | picked candidate has more attention %.2f" % (
          g, len(V), m("img"), m("src"), m("first"), m("second"), np.mean(np.r_[a("pick_first_ab"), a("pick_first_ba")]), m("flip"), cand, pos, pick_more))
# ---- figure: teaser pairs, both orders, attention overlaid
plt.rcParams.update({"font.family": "serif", "font.serif": ["STIXGeneral", "DejaVu Serif"], "font.size": 7.2, "pdf.fonttype": 42})
ROOT = "/media/shawn/C/papers/VLM/judgebench_local/"; lit = lambda v: ast.literal_eval(v) if isinstance(v, str) else v
def sq(p, side=280):
    im = Image.open(ROOT + p).convert("RGB"); w, h = im.size; s = min(w, h); return im.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s)).resize((side, side), Image.LANCZOS)
sel = [i for i in teaser if i in R and len(R[i]) == 2][:3]
fig, axes = plt.subplots(len(sel), 6, figsize=(6.875, 1.22 * len(sel) + .25), gridspec_kw=dict(wspace=.03, hspace=.14, left=.005, right=.995, top=.93, bottom=.01))
for n, i in enumerate(sel):
    x = rows[i]; ctx = lit(x["context"]); cd = lit(x["cand"])
    for oi, o in enumerate(("AB", "BA")):
        r = R[i][o]; Z = np.load(f"{D}/{r['file']}"); paths = [ctx[0]] + (cd if o == "AB" else cd[::-1]); vmax = max(Z[f"m{q}"].max() for q in range(3))
        pick = "first" if r["pA_first"] > .5 else "second"
        for q in range(3):
            ax = axes[n, oi * 3 + q]; im = sq(paths[q]); ax.imshow(im); hm = np.asarray(Image.fromarray((Z[f"m{q}"] / vmax * 255).astype(np.uint8)).resize(im.size, Image.BICUBIC), float) / 255
            ax.imshow(hm, cmap="inferno", alpha=.55 * (hm > .02) + 0.0, vmin=0, vmax=1); ax.set_xticks([]); ax.set_yticks([])
            name = ["source", "shown first", "shown second"][q]; chosen = (q == 1 and pick == "first") or (q == 2 and pick == "second")
            for sp in ax.spines.values(): sp.set_linewidth(1.6 if chosen else .4); sp.set_color("#1f6fd0" if chosen else "#bbbbbb")
            ax.text(.03, .97, name + (" (chosen)" if chosen else ""), transform=ax.transAxes, va="top", fontsize=5.8, color="white", bbox=dict(boxstyle="round,pad=.12", fc=(0, 0, 0, .55), ec="none"))
            ax.text(.97, .03, "%.0f%%" % (100 * r["share"][q] / sum(r["share"])), transform=ax.transAxes, va="bottom", ha="right", fontsize=5.8, color="white", bbox=dict(boxstyle="round,pad=.12", fc=(0, 0, 0, .55), ec="none"))
            if n == 0 and q == 1: ax.set_title(["edit A shown first", "edit B shown first"][oi], fontsize=7.2, pad=2)
fig.savefig(f"{OUT}/F4_attention.pdf", dpi=300); fig.savefig(f"{OUT}/F4_attention.png", dpi=220)
plt.show()

## Tie probability and visual similarity (supplement)
Five judges; slow (loads every edit image).

In [ ]:
fresh()
"""Descriptive (2026-11-14): is the judge's 'tie' a statement about quality or about visual similarity?  ImagenHub edit.
sim = 1 - mean absolute RGB difference between the two edits at 128 x 128 (how alike the two candidates look);
change = mean absolute difference of an edit from its source.  Five open judges.
 1  Spearman correlation of the tie log-odds with sim, among pairs in which both edits failed (human ratings both below the
    midpoint), among decided pairs, and over all pairs; and with 'both failed' (point-biserial via AUROC).
 2  tie-probability AUROC for both-failed vs decided pairs within terciles of sim; and share of human-tied pairs per tercile.
 3  figure F5: examples by a fixed rule (pairs with both ratings 0; hashed order; distinct sources): the three pairs with the
    two edits most alike (top decile of sim) and three with the two edits least alike (bottom decile); Qwen3-VL-8B outputs.
usage (repo root): python tie_similarity.py <out_dir>"""
import sys, os, ast, zlib, json
import numpy as np
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from PIL import Image
import matplotlib
import matplotlib.pyplot as plt
sys.path.insert(0, "/media/shawn/C/papers/EAGLE-VLM"); sys.path.insert(0, "/media/shawn/C/papers/EAGLE-VLM/scripts")
import absolute_quad as AQ
SPEC = "imagenhub/edit"; ROOT = "/media/shawn/C/papers/VLM/judgebench_local/"; lit = lambda v: ast.literal_eval(v) if isinstance(v, str) else v
cfg = AQ.load_cfg(f"configs/exp/{AQ.SPECS[SPEC]}.yaml"); P_ = cfg["paths"]; st = AQ.Store(P_["items"], P_["outputs"], P_.get("items_override"), P_.get("snapshot")); ds, I = AQ.dataset_items(st, SPEC)
cache = {}
def load(p):
    if p not in cache: cache[p] = np.asarray(Image.open(ROOT + p).convert("RGB").resize((128, 128), Image.BILINEAR), float) / 255
    return cache[p]
res = {}
for j in ["qwen3vl8b", "internvl35_8b", "qwen35_9b", "qwen25vl7b", "qwen3vl32b"]:
    S = AQ.signals(st, ds, j, I); ids = sorted(S); PW = AQ.pointwise_feats(st, ds, j, ids)
    Q = (np.array([S[i][2] for i in ids]) + np.array([S[i][3] for i in ids])) / 2; Q = Q / Q.sum(1, keepdims=True); t = np.log(np.maximum(Q[:, 2], 1e-12) / np.maximum(Q[:, 0], Q[:, 1]))
    f1 = np.array([S[i][0] for i in ids]); f2 = np.array([S[i][1] for i in ids]); flip = (f1 > .5) != (f2 > .5)
    R = np.array([AQ.ratings(SPEC, I[i]) for i in ids], float); lab = np.array([str(I[i]["label"]) for i in ids])
    sim = np.array([1 - np.abs(load(lit(I[i]["cand"])[0]) - load(lit(I[i]["cand"])[1])).mean() for i in ids])
    bf = (R < .5).all(1); dec = lab != "tie"; tie = ~dec
    ter = np.digitize(sim, np.quantile(sim, [1 / 3, 2 / 3])); d = dict(n=len(ids))
    d["rho_bothfailed"] = spearmanr(t[bf], sim[bf])[0]; d["rho_decided"] = spearmanr(t[dec], sim[dec])[0]; d["rho_all"] = spearmanr(t, sim)[0]
    d["auc_tie_for_humantie"] = roc_auc_score(tie, t); d["auc_sim_for_humantie"] = roc_auc_score(tie, sim)
    d["auc_tie_for_similar"] = roc_auc_score(ter == 2, t)           # does the tie log-odds find the most-alike third?
    d["flip_by_tercile"] = [float(flip[ter == k].mean()) for k in range(3)]; d["tieprob_median_by_tercile"] = [float(np.median(Q[ter == k, 2])) for k in range(3)]
    d["humantie_by_tercile"] = [float(tie[ter == k].mean()) for k in range(3)]
    d["auc_within_tercile"] = [float(roc_auc_score(tie[ter == k], t[ter == k])) for k in range(3)]
    d["rho_score_sim"] = spearmanr(np.nanmean(PW[:, :2], 1), sim, nan_policy="omit")[0]; d["auc_score_for_bothfailed"] = roc_auc_score(bf[~np.isnan(PW[:, :2]).any(1)], -np.nanmax(PW[~np.isnan(PW[:, :2]).any(1)][:, :2], 1))
    d["auc_score_bothfailed_within_tercile"] = [float(roc_auc_score(bf[(ter == k) & ~np.isnan(PW[:, :2]).any(1)], -np.nanmax(PW[(ter == k) & ~np.isnan(PW[:, :2]).any(1)][:, :2], 1))) for k in range(3)]
    d["auc_tie_bothfailed_within_tercile"] = [float(roc_auc_score(bf[ter == k], t[ter == k])) for k in range(3)]
    res[j] = d
    if j == "qwen3vl8b": keep = dict(ids=ids, Q=Q, f1=f1, f2=f2, PW=PW, R=R, sim=sim, lab=lab)
json.dump(res, open(f"{OUT}/tie_similarity.json", "w"), indent=1)
m = lambda k: np.mean([np.array(res[j][k]) for j in res], 0)
print("mean of 5 judges | Spearman(tie log-odds, similarity of the two edits): both failed %.2f, decided %.2f, all %.2f" % (m("rho_bothfailed"), m("rho_decided"), m("rho_all")))
print("AUROC for human ties: tie log-odds %.3f, pixel similarity alone %.3f | AUROC of the tie log-odds for 'most-alike third' %.3f" % (m("auc_tie_for_humantie"), m("auc_sim_for_humantie"), m("auc_tie_for_similar")))
print("by tercile of similarity (least alike -> most alike): human tie rate", np.round(m("humantie_by_tercile"), 2), "| median tie prob", ["%.1e" % v for v in m("tieprob_median_by_tercile")], "| order flips", np.round(m("flip_by_tercile"), 2), "| tie AUROC for human ties within tercile", np.round(m("auc_within_tercile"), 2))
print("both failed vs rest within tercile: tie log-odds AUROC", np.round(m("auc_tie_bothfailed_within_tercile"), 2), "| pointwise scores AUROC", np.round(m("auc_score_bothfailed_within_tercile"), 2), "| Spearman(mean score, similarity) %.2f" % m("rho_score_sim"))
# ---- figure
K = keep; ids = K["ids"]; z = (K["R"].max(1) == 0) & ~np.isnan(K["PW"][:, :2]).any(1); lo, hi = np.quantile(K["sim"][z], [.1, .9])
order = sorted(np.flatnonzero(z), key=lambda k: zlib.crc32(ids[k].encode()))
def pick(cond):
    out, seen = [], set()
    for k in order:
        s = lit(I[ids[k]]["context"])[0]
        if cond(k) and s not in seen: out.append(k); seen.add(s)
        if len(out) == 3: break
    return out
G = [("Both edits failed and look alike", pick(lambda k: K["sim"][k] >= hi)), ("Both edits failed and look different", pick(lambda k: K["sim"][k] <= lo))]
plt.rcParams.update({"font.family": "serif", "font.serif": ["STIXGeneral", "DejaVu Serif"], "font.size": 7.2, "pdf.fonttype": 42})
def sq(p, side=260):
    im = Image.open(ROOT + p).convert("RGB"); w, h = im.size; s = min(w, h); return im.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s)).resize((side, side), Image.LANCZOS)
fig = plt.figure(figsize=(6.875, 3.75)); outer = fig.add_gridspec(1, 2, left=.004, right=.996, top=.925, bottom=.01, wspace=.045)
for g, (title, ks) in enumerate(G):
    sub = outer[g].subgridspec(3, 3, wspace=.03, hspace=.36); fig.text(.25 + .5 * g, .972, title, ha="center", va="center", fontsize=8.2, fontweight="bold", color="#1a9f6e" if g == 0 else "#d9541e")
    for r, k in enumerate(ks):
        x = I[ids[k]]; ctx = lit(x["context"]); cd = lit(x["cand"])
        for c, (p, nm) in enumerate(((ctx[0], "source"), (cd[0], "edit A"), (cd[1], "edit B"))):
            ax = fig.add_subplot(sub[r, c]); ax.imshow(sq(p)); ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values(): sp.set_linewidth(.4); sp.set_color("#bbbbbb")
            ax.text(.03, .97, nm, transform=ax.transAxes, va="top", fontsize=5.8, color="white", bbox=dict(boxstyle="round,pad=.12", fc=(0, 0, 0, .55), ec="none"))
            if c == 0:
                ins = x["question"].strip(); ax.set_title("“" + (ins if len(ins) <= 40 else ins[:38] + "…") + "”", fontsize=6.6, loc="left", pad=1.5, style="italic")
            if c == 1:
                a1 = "A" if K["f1"][k] > .5 else "B"; a2 = "A" if K["f2"][k] > .5 else "B"
                ax.set_xlabel(f"tie prob. {K['Q'][k, 2]:.0e}   choice: {a1} / {a2}" + ("  (flips)" if a1 != a2 else "") + f"   scores {K['PW'][k, 0]:.1f}, {K['PW'][k, 1]:.1f}", fontsize=6.0, labelpad=1.5, color="#111")
fig.savefig(f"{OUT}/F5_tie_is_similarity.pdf", dpi=300); fig.savefig(f"{OUT}/F5_tie_is_similarity.png", dpi=220)
print("example pairs:", [[ids[k] for k in ks] for _, ks in G], "| tie prob:", [[float("%.1e" % K["Q"][k, 2]) for k in ks] for _, ks in G])
plt.show()

## Coverage at three error targets (addendum 22) — light
Reads `data/multi_alpha.json` (written by `results/addendum22/multi_alpha.py` from the finished certification runs). One panel per benchmark; a filled marker keeps the promised accuracy, a hollow one breaks it. Each baseline is shown in its most favourable variant.

In [ ]:
fresh()
import os, json
import numpy as np
import matplotlib.pyplot as plt
from figstyle import C, setup, DATA
setup()
R = json.load(open(os.path.join(DATA, "multi_alpha.json")))
SHORT = {"MLLM-as-a-Judge (pairs)": "MLLM-J pairs", "MLLM-as-a-Judge (scores)": "MLLM-J scores", "MT-Bench (text)": "MT-Bench (text)", "Chatbot Arena (text)": "Chatbot Arena (text)",
         "VisionArena (battles)": "VisionArena", "GenAI-Bench": "GenAI-Bench", "GenAI-Arena gen.": "GenAI-Arena gen.", "GenAI-Arena edit": "GenAI-Arena edit", "ImagenHub edit": "ImagenHub edit"}
names = sorted([n for n in R if n in SHORT], key=lambda n: -R[n]["tie_rate"])            # most ties first, as in the main table
# ---- editable: which methods are drawn, their labels, colours and line styles
METH = [("ours", "Ours", C["ours"], "-", "o"), ("tieaware", "Best tie-aware baseline", C["pointwise"], "-", "s"),
        ("scope", "SCOPE / LEC rule", C["baseline"], "--", "^"), ("toe", "Trust-or-Escalate", C["ink2"], ":", "v")]
AL = ["0.1", "0.2", "0.3"]; xs = [.1, .2, .3]
fig, axes = plt.subplots(2, 5, figsize=(6.875, 2.75), sharey=True, sharex=True, gridspec_kw=dict(wspace=.08, hspace=.42, left=.065, right=.995, top=.91, bottom=.13))
for ax, n in zip(axes.ravel(), names):
    for key, lab, col, ls, mk in METH:
        v = [R[n]["alpha"][a][key] for a in AL]; cov = [x["cov"] if x["cov"] == x["cov"] else 0 for x in v]
        ax.plot(xs, cov, ls=ls, color=col, lw=1.1, zorder=2)
        for x_, c_, vv in zip(xs, cov, v):
            keeps = bool(vv.get("keeps")) and c_ >= .005; ax.plot(x_, c_, mk, color=col, mfc=col if keeps else "white", mew=.9, ms=3.6, zorder=3)   # hollow: promise broken or nothing answered
    ax.set_title(f"{SHORT[n]}  ({100 * R[n]['tie_rate']:.0f}% ties)", fontsize=6.4, fontweight="normal", pad=2.5)
    ax.set_xticks(xs); ax.set_xticklabels([".1", ".2", ".3"]); ax.set_ylim(-.05, 1.05); ax.set_yticks([0, .5, 1]); ax.tick_params(labelsize=6, length=2, pad=1.5); ax.set_xlim(.07, .33)
ax = axes.ravel()[-1]; ax.axis("off")
for key, lab, col, ls, mk in METH: ax.plot([], [], ls=ls, marker=mk, color=col, lw=1.1, ms=3.6, label=lab)
ax.plot([], [], "o", color=C["ink2"], ms=3.6, ls="", label="keeps the promise"); ax.plot([], [], "o", color=C["ink2"], mfc="white", mew=.9, ms=3.6, ls="", label="breaks it, or silent")
ax.legend(loc="center left", bbox_to_anchor=(-.12, .5), fontsize=5.8, handlelength=2.0, labelspacing=.45, borderaxespad=0)
fig.text(.53, .015, r"error target $\alpha$ (promised accuracy $1-\alpha$)", ha="center", fontsize=6.8, color=C["ink2"]); fig.text(.008, .52, "share of sources answered", rotation=90, va="center", fontsize=6.8, color=C["ink2"])
fig.savefig(f"{OUT}/F8_multi_alpha.pdf", bbox_inches=None); fig.savefig(f"{OUT}/F8_multi_alpha.png", dpi=240, bbox_inches=None)
plt.show()


## Main table (a) as a line figure: accuracy and coverage against the share of ties, three error targets — light

In [ ]:
fresh()
import os, json
import numpy as np
import matplotlib.pyplot as plt
from figstyle import C, setup, DATA
setup()
# Main table (a) as a figure: every benchmark is one x position (its share of human ties); one column per error target.
R = json.load(open(os.path.join(DATA, "multi_alpha.json")))
names = sorted([n for n in R if "Best-of-8" not in n], key=lambda n: R[n]["tie_rate"]); x = np.array([R[n]["tie_rate"] for n in names])
def get(n, a, key):                      # -> (coverage, accuracy among answers, keeps the promise)
    if key in ("scope_dropped", "toe_dropped"):
        v = [r for r in R[n]["alpha"][a][key.split("_")[0] + "_variants"] if "ties dropped" in r[0]][0]; cov, risk = v[1], v[2]
    else:
        v = R[n]["alpha"][a][key]; cov, risk = v["cov"], v["risk"]
    cov = cov if cov == cov else 0.0; acc = 1 - risk if (risk == risk and cov >= .005) else np.nan
    return cov, acc, (acc == acc) and acc >= 1 - float(a) - 1e-9
# ---- editable: methods (key, label, colour, line style, marker), titles, axis labels
METH = [("ours", "Ours (ties kept)", C["ours"], "-", "o"), ("tieaware", "Best tie-aware baseline (ties kept)", C["pointwise"], "-.", "s"),
        ("scope_dropped", "SCOPE / LEC rule (ties dropped)", C["baseline"], "--", "^"), ("toe_dropped", "Trust-or-Escalate (ties dropped)", C["ink2"], ":", "v")]
AL = ["0.1", "0.2", "0.3"]
ROWS = [(1, "Accuracy among answers", "accuracy among answers"), (0, "Share of sources answered", "share of sources answered")]
XLAB = "share of pairs that people rate as ties (one benchmark per marker)"
fig, axes = plt.subplots(2, 3, figsize=(6.875, 3.35), sharex=True, sharey="row", gridspec_kw=dict(wspace=.07, hspace=.30, left=.07, right=.995, top=.84, bottom=.125))
for r, (idx, title, ylab) in enumerate(ROWS):
    for c, a in enumerate(AL):
        ax = axes[r, c]
        if idx == 1:
            ax.axhline(1 - float(a), color=C["ink2"], lw=.7, ls=(0, (4, 2)), zorder=1)
            ax.text(.185, 1 - float(a) + (.012 if a != "0.3" else -.012), f"promised {1 - float(a):.1f}".replace("0.", "."), ha="left", va="bottom" if a != "0.3" else "top", fontsize=5.6, color=C["ink2"])
        for key, lab, col, ls, mk in METH:
            V = [get(n, a, key) for n in names]; y = np.array([v[idx] for v in V], float)
            ax.plot(x, y, ls=ls, color=col, lw=1.1, zorder=2)
            for x_, y_, v in zip(x, y, V):
                if y_ == y_: ax.plot(x_, y_, mk, color=col, mfc=col if v[2] else "white", mew=.9, ms=3.6, zorder=3)     # hollow: promise broken, or nothing answered
        ax.set_title(f"({'abcdef'[3 * r + c]}) {title}, $\\alpha={a[1:]}$", fontsize=6.6, fontweight="normal", pad=3)
        ax.set_ylim((.3, 1.02) if idx == 1 else (-.05, 1.05)); ax.set_xlim(.18, .58); ax.set_xticks([.2, .3, .4, .5]); ax.set_xticklabels([".2", ".3", ".4", ".5"]); ax.tick_params(labelsize=6, length=2, pad=1.5)
    axes[r, 0].set_ylabel(ylab, fontsize=6.6, color=C["ink2"], labelpad=2)
h = [plt.Line2D([], [], ls=ls, marker=mk, color=col, lw=1.1, ms=3.6, label=lab) for key, lab, col, ls, mk in METH]
h += [plt.Line2D([], [], ls="", marker="o", color=C["ink2"], ms=3.6, label="keeps the promise"), plt.Line2D([], [], ls="", marker="o", color=C["ink2"], mfc="white", mew=.9, ms=3.6, label="breaks it, or silent")]
fig.legend(handles=h, loc="upper center", ncol=3, fontsize=5.9, handlelength=2.2, columnspacing=1.4, frameon=False, bbox_to_anchor=(.53, 1.005))
fig.text(.53, .012, XLAB, ha="center", fontsize=6.8, color=C["ink2"])
fig.savefig(f"{OUT}/F9_main_lines.pdf", bbox_inches=None); fig.savefig(f"{OUT}/F9_main_lines.png", dpi=240, bbox_inches=None)
plt.show()


## Main table (b) as a figure: three candidate designs (for choosing; not in the paper yet) — light

In [ ]:
fresh()
import numpy as np
import matplotlib.pyplot as plt
from figstyle import C, setup
setup()
# Main table (b) as a figure: three candidate designs.  Numbers: tab:decisionsfull (alpha = .2, per 100 test sources).
# (correct ships, correct rejects, error rate of ships, error rate of rejects); wrong = correct * e / (1 - e).
DATA_B = {
    "ImagenHub edit": dict(pi=.74, ceil=33, rows={"Ship only": (0.0, 0, 0, 0), "Ship or reject:\ncomparison": (2.1, 38.0, .37, .13), "Ship or reject:\npointwise": (3.3, 47.7, .38, .12), "Ship or reject:\nall (ours)": (5.8, 51.6, .33, .11)}),
    "GenAI-Bench, top quarter": dict(pi=.48, ceil=65, rows={"Ship only": (5.2, 0, .20, 0), "Ship or reject:\ncomparison": (1.3, 3.4, .25, .15), "Ship or reject:\npointwise": (16.8, 22.6, .21, .15), "Ship or reject:\nall (ours)": (18.0, 22.9, .21, .15)}),
    "Web-agent runs": dict(pi=.63, ceil=46, rows={"Ship only": (0.9, 0, .19, 0), "Ship or reject:\ncomparison": (0.3, 2.9, .26, .17), "Ship or reject:\npointwise": (13.0, 44.7, .27, .12), "Ship or reject:\nall (ours)": (16.7, 47.0, .23, .13)})}
def parts(r):
    cs, cr, es, er = r; ws = cs * es / (1 - es) if es else 0; wr = cr * er / (1 - er) if er else 0; return cs, cr, ws + wr, max(0, 100 - cs - cr - ws - wr)
# ---- editable: colours and labels
COL = dict(ship=C["succeed"], reject=C["ours"], wrong=C["fail"], esc="#e9e8e2"); LAB = ["correct: ship", "correct: reject both", "wrong decision", "escalated to a person"]
POL = list(next(iter(DATA_B.values()))["rows"]); short = [p.replace("Ship or reject:\n", "+ reject, ") for p in POL]
fig = plt.figure(figsize=(6.875, 6.3)); gs = fig.add_gridspec(3, 3, hspace=.62, wspace=.10, left=.17, right=.985, top=.93, bottom=.06)
for c, (name, D) in enumerate(DATA_B.items()):
    P = np.array([parts(D["rows"][p]) for p in POL]); tot = P[:, 0] + P[:, 1]
    # ---- option A: what happens to 100 sources (stacked)
    ax = fig.add_subplot(gs[0, c]); y = np.arange(len(POL))[::-1]; left = np.zeros(len(POL))
    for k, key in enumerate(("ship", "reject", "wrong", "esc")):
        ax.barh(y, P[:, k], left=left, color=COL[key], height=.62, edgecolor="white", lw=.6, label=LAB[k] if c == 0 else None); left += P[:, k]
    for yy, t in zip(y, tot): ax.text(t + P[yy == y][0][2] + 2, yy, f"{t:.0f}" if t >= 1 else f"{t:.1f}", va="center", fontsize=6.4, fontweight="bold" if yy == 0 else "normal")
    ax.set_yticks(y); ax.set_yticklabels(short if c == 0 else [], fontsize=6.4); ax.set_xlim(0, 100); ax.grid(axis="y", visible=False)
    ax.set_title(f"{name}  ($\\pi={D['pi']:.2f}$)".replace("0.", "."), fontsize=6.8, fontweight="normal", pad=3); ax.tick_params(labelsize=6, length=2)
    if c == 0: ax.text(-.30, 1.16, "Option A: what happens to 100 sources", transform=ax.transAxes, fontsize=7.6, fontweight="bold")
    if c == 1: ax.set_xlabel("test sources (of 100)", fontsize=6.6)
    # ---- option B: correct decisions as bars split by action, wrong ones below zero
    ax = fig.add_subplot(gs[1, c]); x = np.arange(len(POL))
    ax.bar(x, P[:, 1], color=COL["reject"], width=.62, edgecolor="white", lw=.6); ax.bar(x, P[:, 0], bottom=P[:, 1], color=COL["ship"], width=.62, edgecolor="white", lw=.6)
    ax.bar(x, -P[:, 2], color=COL["wrong"], width=.62, edgecolor="white", lw=.6); ax.axhline(0, color=C["ink2"], lw=.6)
    for xx, t in zip(x, tot): ax.text(xx, t + 2, f"{t:.0f}" if t >= 1 else f"{t:.1f}", ha="center", fontsize=6.4, fontweight="bold" if xx == len(POL) - 1 else "normal")
    ax.set_xticks(x); ax.set_xticklabels(["ship\nonly", "+ reject\ncompar.", "+ reject\npointw.", "+ reject\nall (ours)"], fontsize=5.8); ax.set_ylim(-22, 78); ax.grid(axis="x", visible=False)
    ax.set_yticks([-20, 0, 20, 40, 60]); ax.set_yticklabels(["20", "0", "20", "40", "60"] if c == 0 else []); ax.tick_params(labelsize=6, length=2)
    if c == 0: ax.set_ylabel("wrong  |  correct decisions per 100", fontsize=6.4); ax.text(-.30, 1.08, "Option B: correct above the line, wrong below", transform=ax.transAxes, fontsize=7.6, fontweight="bold")
    # ---- option C: one line per benchmark from ship-only to ours, with the ceiling of a ship-only rule
    ax = fig.add_subplot(gs[2, c])
    ax.axhspan(0, D["ceil"], color="#f1f0ea", zorder=0); ax.text(3.45, D["ceil"] - 1.5, f"ceiling of any ship-only rule: {D['ceil']}", ha="right", va="top", fontsize=5.8, color=C["ink2"])
    ax.plot(x, tot, "-", color=C["ours"], lw=1.3, zorder=2); ax.plot(x[:-1], tot[:-1], "o", color="white", mec=C["ours"], mew=1.1, ms=5, zorder=3); ax.plot(x[-1], tot[-1], "o", color=C["ours"], ms=6, zorder=3)
    for xx, t in zip(x, tot): ax.text(xx, t + 4, f"{t:.0f}" if t >= 1 else f"{t:.1f}", ha="center", fontsize=6.4, fontweight="bold" if xx == len(POL) - 1 else "normal")
    ax.set_xticks(x); ax.set_xticklabels(["ship\nonly", "+ reject\ncompar.", "+ reject\npointw.", "+ reject\nall (ours)"], fontsize=5.8); ax.set_ylim(0, 78); ax.set_xlim(-.45, 3.5); ax.grid(axis="x", visible=False)
    ax.set_yticks([0, 20, 40, 60]); ax.set_yticklabels(["0", "20", "40", "60"] if c == 0 else []); ax.tick_params(labelsize=6, length=2)
    if c == 0: ax.set_ylabel("correct decisions per 100", fontsize=6.4); ax.text(-.30, 1.08, "Option C: one line per setting, against the ship-only ceiling", transform=ax.transAxes, fontsize=7.6, fontweight="bold")
h = [plt.Rectangle((0, 0), 1, 1, color=COL[k]) for k in ("ship", "reject", "wrong", "esc")]
fig.legend(h, LAB, loc="upper right", ncol=4, fontsize=6.2, frameon=False, bbox_to_anchor=(.99, 1.0), handlelength=1.2, columnspacing=1.2)
fig.savefig(f"{OUT}/F10_decision_options.pdf", bbox_inches=None); fig.savefig(f"{OUT}/F10_decision_options.png", dpi=220, bbox_inches=None)
plt.show()


## Qualitative figure (supplement): certified decisions on real pairs — needs the data cell c04

In [ ]:
fresh(EDIT['RC_V3']); globals().update(EDIT)
# ---------------------------------------------------------------- Qualitative figure (supplement, fig:qual): certified decisions on real pairs
# One judge (Qwen3-VL-8B), ImagenHub edit, the first source split of the paper's protocol (25 % fit, 50 % calibration, 25 % test),
# alpha = .2, ship-or-reject policy on all signals.  One test pair per panel, the first in crc32 order with a new source:
#  (1) certified "reject both", correct   (2) certified "ship", correct   (3) certified decision that is wrong   (4) escalated to a person
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from eaglevlm.pipeline import certify as CE
AL_Q = .2; n = len(ids); p_ = (f1 + f2) / 2
lo_, hi_ = AQ.SCALE[SPEC.split("/")[0]]; RR = np.array([AQ.ratings(SPEC, I[i]) for i in ids], float); acc = RR >= lo_ + .5 * (hi_ - lo_) - 1e-9
anyok = acc[:, 0] | acc[:, 1]; y4 = np.where(acc[:, 0] & acc[:, 1], 0, np.where(acc[:, 0], 1, np.where(acc[:, 1], 2, 3)))
ug, gi = np.unique(np.array([source_key(I[i]) for i in ids]), return_inverse=True); rng = np.random.default_rng(0)
fit, cal, te = AQ.splits(n, gi, {"unit": "source1", "split": {"fit": .25, "cal": .5}}, rng)
pw = np.where(np.isnan(PW), np.nanmean(PW[fit], 0)[None, :], PW)
Fb = np.c_[np.log(np.maximum(p_, 1e-9) / np.maximum(1 - p_, 1e-9)), CE.H(p_), np.abs(p_ - .5), ((f1 > .5) == (f2 > .5)).astype(float)]
F = np.c_[Fb, np.log(Q + 1e-9), pw]; sc = StandardScaler().fit(F[fit]); m_ = LogisticRegression(max_iter=5000).fit(sc.transform(F[fit]), y4[fit])
P4 = np.full((n, 4), 1e-6); P4[:, m_.classes_] = m_.predict_proba(sc.transform(F)); P4 /= P4.sum(1, keepdims=True)
a_, s_ = AQ.decide(P4[:, 0] + P4[:, 1], P4[:, 0] + P4[:, 2], P4[:, 3], "sr"); ship = a_ < 2; kk = np.minimum(a_, 1)
err = np.where(ship, ~acc[np.arange(n), kk], anyok).astype(float); br = np.where(ship, 0, 1)
lam = AQ.fixed_seq(s_, err, br, "sr", AL_Q, "all", cal, fit); okd = s_ <= lam; T = np.zeros(n, bool); T[te] = True
print("qual: test pairs %d | decided %.3f | wrong among decided %.3f | rejects %.3f of decided" % (T.sum(), okd[T].mean(), err[T & okd].mean(), (~ship)[T & okd].mean()))
crc = np.argsort([zlib.crc32(i.encode()) for i in ids]); okpw = ~np.isnan(PW[:, :2]).any(1)
M = [T & okd & ~ship & (err == 0) & okpw, T & okd & ship & (err == 0) & okpw, T & okd & (err == 1) & okpw, T & ~okd & okpw]
seen = set(src[ids.index(d)] for d in json.load(open(f"{OUT}/F1_pairs_four.json")) if d in ids) if os.path.exists(f"{OUT}/F1_pairs_four.json") else set()   # not the pairs of the teaser
pick = []
for m in M:
    for k in crc:
        if m[k] and src[k] not in seen:
            pick.append(k); seen.add(src[k]); break
print("qual candidates per panel", [int(m.sum()) for m in M], "| picked", [ids[k] for k in pick])
json.dump([dict(id=ids[k], action=int(a_[k]), certified=bool(okd[k]), wrong=bool(err[k]), ratings=RR[k].tolist(), p4=P4[k].round(4).tolist()) for k in pick], open(f"{OUT}/F11_qual_pairs.json", "w"), indent=1)
# ---- editable: headers and colours
HEAD = ["(1) Certified: reject both (correct)", "(2) Certified: ship (correct)", "(3) Certified, but wrong", "(4) Not certified: escalated"]
HCOL = [C["ok"], C["ok"], C["fail"], C["grey"]]; ACT = ["ship A", "ship B", "reject both"]
fig = plt.figure(figsize=(6.875, 3.95)); gs = fig.add_gridspec(2, 2, left=.005, right=.995, top=.995, bottom=.012, wspace=.035, hspace=.075)
for n_, k in enumerate(pick):
    x = I[ids[k]]; ctx = lit(x["context"]); cd = lit(x["cand"]); sub = gs[n_ // 2, n_ % 2].subgridspec(3, 3, height_ratios=[.21, 1, .46], wspace=.03, hspace=.02)
    ax = fig.add_subplot(sub[0, :]); ax.axis("off"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.text(0, .78, HEAD[n_], fontsize=7.6, fontweight="bold", color=HCOL[n_], va="center")
    ax.text(1, .78, f"people: A {RR[k, 0]:.2f}, B {RR[k, 1]:.2f} (acceptable from {lo_ + .5 * (hi_ - lo_):.1f})", fontsize=6.1, color=C["ink2"], va="center", ha="right")
    ins = x["question"].strip(); ins = ins if len(ins) <= 60 else ins[:58] + "…"; ax.text(.004, .2, "instruction: “" + ins + "”", fontsize=6.9, style="italic", color=C["ink"], va="center")
    for c, (p, t) in enumerate(((ctx[0], "source"), (cd[0], "edit A"), (cd[1], "edit B"))):
        ax = fig.add_subplot(sub[1, c]); ax.imshow(img(p)); ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_visible(True); sp.set_linewidth(.5 if c == 0 else 1.3); sp.set_color(C["light"] if c == 0 else (C["ok"] if acc[k, c - 1] else C["fail"]))
        ax.text(.03, .97, t if c == 0 else t + (": acceptable" if acc[k, c - 1] else ": not acceptable"), transform=ax.transAxes, va="top", ha="left", fontsize=6.2, color="white", bbox=dict(boxstyle="round,pad=.15", fc=(0, 0, 0, .55), ec="none"))
    ax = fig.add_subplot(sub[2, :]); ax.axis("off"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.text(0, .86, f"Judge:  scores A {PW[k, 0]:.1f}, B {PW[k, 1]:.1f}   “which is better?” A {p_[k]:.2f}   probability of “tie” " + ("below 0.01" if Q[k, 2] < .005 else f"{Q[k, 2]:.2f}"), fontsize=6.5, color=C["ink"], va="center")
    ax.text(0, .58, f"Head:  both acceptable {P4[k, 0]:.2f}   only A {P4[k, 1]:.2f}   only B {P4[k, 2]:.2f}   neither {P4[k, 3]:.2f}", fontsize=6.5, color=C["ours"], va="center")
    out = ("certified" if okd[k] else "below the certified threshold: no decision") + ("" if not okd[k] else (" — correct" if err[k] == 0 else " — wrong: " + ("the shipped edit is not acceptable" if ship[k] else "one edit is acceptable")))
    ax.text(0, .30, f"Action:  {ACT[a_[k]]}   ({out})", fontsize=6.6, fontweight="bold", color=HCOL[n_], va="center")
fig.savefig(f"{OUT}/F11_qualitative.pdf"); fig.savefig(f"{OUT}/F11_qualitative.png", dpi=220)
plt.show()


## Decisions figure (main text): correct above the axis, wrong below, with source-bootstrap intervals — needs the data cell c04 only for the serif style

In [ ]:
fresh(EDIT['RC_V3'])          # same serif style as Figures 1-3
import os, json
import numpy as np
import matplotlib.pyplot as plt
from figstyle import DATA
# Decisions (main text).  Above the axis: correct certified decisions per 100 test sources, split into "reject both" and "ship".
# Below the axis: wrong decisions.  Whiskers: 95 % source-bootstrap intervals (decision_ci.py -> data/decision_ci.json).
R = json.load(open(os.path.join(DATA, "decision_ci.json")))
# ---- editable: colours, labels, titles
COL = dict(reject="#1f5fb4", ship="#8fd3a8", wrong="#d9541e", ink="#111111", ink2="#55534f", grid="#e4e2dc", band="#f6f5f1")
TICKS = ["Winner-only\n(ship only)", "comparison", "scores", "both"]
TITLE = {"ImagenHub edit": "(a) ImagenHub edit", "GenAI-Bench, top quarter": "(b) GenAI-Bench, top quarter", "Web-agent runs": "(c) Web-agent runs"}
fig, axes = plt.subplots(1, 3, figsize=(6.875, 2.15), sharey=True, gridspec_kw=dict(wspace=.05, left=.066, right=.997, top=.83, bottom=.215))
for ax, (name, D) in zip(axes, R.items()):
    P = list(D["policies"].values()); x = np.arange(len(P)); w = .58
    rj = np.array([p["reject"][0] for p in P]); sh = np.array([p["ship"][0] for p in P]); tot = np.array([p["correct"] for p in P]); wr = np.array([p["wrong"] for p in P])
    ax.axhspan(-30, 0, color=COL["band"], zorder=0); ax.axvspan(.5, 3.5, ymin=0, ymax=1, color="none")
    for xi in x:
        last = False; al = 1.0 if xi > 0 else .78          # the three ship-or-reject bars are all ours; no bar is singled out
        ax.bar(xi, rj[xi], width=w, color=COL["reject"], alpha=al, zorder=2, lw=0); ax.bar(xi, sh[xi], bottom=rj[xi], width=w, color=COL["ship"], alpha=al, zorder=2, lw=0)
        ax.bar(xi, -wr[xi, 0], width=w, color=COL["wrong"], alpha=al, zorder=2, lw=0)
        if last: ax.bar(xi, tot[xi, 0] + wr[xi, 0], bottom=-wr[xi, 0], width=w, fill=False, ec=COL["ink"], lw=.9, zorder=4)
    ek = dict(fmt="none", ecolor=COL["ink"], elinewidth=.7, capsize=1.6, capthick=.7, zorder=5)
    m = tot[:, 0] > .05; ax.errorbar(x[m], tot[m, 0], yerr=[tot[m, 0] - tot[m, 1], tot[m, 2] - tot[m, 0]], **ek); ax.errorbar(x[m], -wr[m, 0], yerr=[wr[m, 2] - wr[m, 0], wr[m, 0] - wr[m, 1]], **ek)
    for xi, t in zip(x, tot):
        ax.text(xi, t[2] + 2.0, f"{t[0]:.0f}" if t[0] >= 1.5 else f"{t[0]:.1f}", ha="center", va="bottom", fontsize=7.0 if xi > 0 else 6.8, fontweight="bold" if xi > 0 else "normal", color=COL["ink"])
    ax.axhline(0, color=COL["ink"], lw=.6, zorder=3); ax.axvline(.5, color=COL["ink2"], lw=.5, ls=(0, (2, 2)), zorder=1)
    ax.plot([.62, 3.38], [-23.5, -23.5], color=COL["ink"], lw=.7, zorder=3); ax.plot([.62, .62], [-23.5, -21.5], color=COL["ink"], lw=.7, zorder=3); ax.plot([3.38, 3.38], [-23.5, -21.5], color=COL["ink"], lw=.7, zorder=3)
    ax.text(2, -24.3, "ours: ship or reject, reading", ha="center", va="top", fontsize=6.3, color=COL["ink"], fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(TICKS, fontsize=6.5); ax.set_xlim(-.5, 3.5)
    ax.set_ylim(-30, 80); ax.set_yticks([-20, 0, 20, 40, 60, 80]); ax.set_yticklabels(["20", "0", "20", "40", "60", "80"], fontsize=6.5)
    ax.grid(axis="y", color=COL["grid"], lw=.5, zorder=0); ax.grid(axis="x", visible=False); ax.tick_params(length=0, pad=2); ax.set_axisbelow(True)
    for s_ in ("top", "right", "left", "bottom"): ax.spines[s_].set_visible(False)
    ax.set_title(TITLE.get(name, name), fontsize=7.6, fontweight="bold", loc="left", pad=4)
    ax.text(1, 1.035, f"$\\pi={D['pi']:.2f}$".replace("0.", "."), transform=ax.transAxes, ha="right", va="bottom", fontsize=6.6, color=COL["ink2"])
axes[0].set_ylabel("certified decisions\nper 100 sources", fontsize=6.9, labelpad=3, color=COL["ink"])
axes[0].text(-.47, 74, "correct", fontsize=6.2, color=COL["reject"], va="top", fontweight="bold"); axes[0].text(-.47, -3, "wrong", fontsize=6.2, color=COL["wrong"], va="top", fontweight="bold")
h = [plt.Rectangle((0, 0), 1, 1, color=COL[k], lw=0) for k in ("reject", "ship", "wrong")] + [plt.Line2D([], [], color=COL["ink"], lw=.7, marker="_", ms=4)]
fig.legend(h, ["correct: reject both", "correct: ship", "wrong decision", "95% interval (bootstrap over sources)"], loc="upper center", ncol=4, fontsize=6.4, frameon=False, bbox_to_anchor=(.53, 1.015), handlelength=1.3, columnspacing=1.5, handletextpad=.5)
fig.savefig(f"{OUT}/F12_decisions.pdf", bbox_inches=None); fig.savefig(f"{OUT}/F12_decisions.png", dpi=260, bbox_inches=None)
plt.show()
